GALLAPONT Thibault - MALLE Arnaud - MUEL Thomas

# Projet de NLP : peut-on battre notre meilleure pipeline ?

**Baseline** : notre meilleure pipeline actuelle, `cardiffnlp/twitter-xlm-roberta-large-2022` entraîné deux fois (étiquettes hard et soft), moyenne des deux probabilités ; **F1 YES = 0,862 sur le dev** au dernier run.

Ce notebook ne reprend pas tout le rapport. Il garde cette baseline et ne teste que les idées dont on attend un gain réel, chacune mesurée contre la baseline :

| Partie | Idée | Pourquoi on s'attend à un gain |
|---|---|---|
| 0 | analyse des erreurs du dernier run | identifie où gagner |
| 2 | **preprocessing corrigé** | 5 bugs trouvés dans le nettoyage actuel |
| 3, partout | **seuil choisi sur l'OOF**, **ensemble des 5 modèles de folds** | le modèle soft est sous-confiant (0,865 → 0,875 avec un seuil plus bas) ; moyenner des modèles est le gain le plus sûr |
| 4 | **« premier modèle » bon marché** (TF-IDF, **prior du mot-clé de collecte**, prior kNN, features) combiné à BERT | signal complémentaire (sujet du tweet), corrélé à 0,57 avec la part de YES |
| 6 | **sortie multi-classes vs binaire** (4 intentions, 7 niveaux de votes, multi-tâches) | la moitié des erreurs sont des tweets qui *rapportent* ou *dénoncent* le sexisme |
| 6 | **AEDA**, **tête MultiCNN** | demandés ; testés proprement cette fois |
| 7 | meilleure combinaison des réglages, **3 seeds** | cumul des gains |
| 8 | **LLM 7B** : few-shot, justifications générées (CIMAT-GTO, 2e en 2025), **fine-tuning LoRA** (Mario, 1er en 2025), distillation vers XLM-R | les meilleurs systèmes d'EXIST 2025 utilisent un LLM ; il comprend l'ironie et la définition EXIST |
| 9 | **combinaisons** : ensemble, stacking, cascade « modèle 1 sûr, sinon BERT », routage des cas incertains vers le LLM, seuil par langue | erreurs différentes → gains en les combinant |

**Protocole.** Chaque configuration est entraînée en **validation croisée à 5 folds** sur le train : on obtient une prédiction *out-of-fold* (OOF) pour chacun des 6 920 tweets, et la prédiction du dev est la **moyenne des 5 modèles**. Tous les choix (époque, seuil, réglages, poids des combinaisons) se font **sur l'OOF** ; le dev ne sert qu'à mesurer, avec un intervalle de confiance et un test apparié contre la baseline.

**Exécution.** Colab avec un **G4** (RTX PRO 6000 Blackwell, 96 Go). Tout est mis en cache sur Drive (`cache_modelling2/`) après chaque fold : si la session tombe, on relance tout et le notebook reprend où il s'était arrêté. Durées indicatives :

| Partie | Durée |
|---|---|
| 4 : modèle 1, embeddings | ~10 min |
| 6 : baseline (2 calendriers × hard/soft) + 6 ablations | ~2 h |
| 7 : combinaison, 3 seeds + hard | ~1 h |
| 8 : XLM-R XL, soft + hard | ~2,5 h |
| 9 : LLM few-shot, justifications + XLM-R, LoRA + distillation | ~2 h |
| **Total** | **~7-8 h** (lancer en arrière-plan) |

## 0. Ce que montrent les erreurs du run précédent

Analyse de `dev_predictions.csv` (XLM-R large, 1 époque, LR 2e-5) :

| | F1 YES (seuil 0,5) | meilleur seuil (lu sur le dev, indicatif) |
|---|---|---|
| hard | 0,862 | 0,863 |
| soft | 0,865 | **0,875** (seuil 0,45) |
| moyenne | 0,862 | 0,868 |

- **Anglais 0,835, espagnol 0,882.**
- **127 erreurs : 68 FP, 59 FN.** Elles se concentrent sur les cas ambigus : 41 FP sur 116 tweets à 2 votes YES (35 %), 35 FN sur 144 tweets à 4 votes (24 %), contre 6 FP sur 203 tweets à 0 vote.
- **Le modèle soft est sous-confiant** : son meilleur seuil est sous 0,5. Il faut choisir le seuil (sur l'OOF).

**Typologie des faux négatifs**

1. **Tweets qui *parlent* du sexisme sur un ton neutre ou informatif** (REPORTED / JUDGEMENTAL), que les annotateurs comptent comme sexistes : « Barreras invisibles para madres trabajadoras » (p = 0,09), brecha salarial, interseccionalidad, doble discriminación, « everyday sexism » session, statistiques sur le harcèlement, *superwoman syndrome*, *phallocentrism*. Le modèle a appris que les annonces, les campagnes et les URL sont « NO ».
2. **Expériences personnelles** : « un señor en la calle me gritó algo », « sufro las secuelas de tanto abuso », « harassed for pro choice ».
3. **Ironie et antiféminisme implicite** : « #YoSiTeCreo … absuelto, quién le devuelve su vida », « Debanhi … se violó sola », « Democracia paritaria … pura mierda ».
4. **Insultes genrées employées « affectueusement » ou sur soi** : « por ZORRAS!! 💖 », « slutbag », « llorar como una nenaza ».

**Typologie des faux positifs**

1. **Vulgarité ou insultes qui ne visent pas les femmes** : « pareces una puta vaca pastando », « penis envy idiots », « me gustaría follar más ».
2. **Tweets positifs sur les femmes** : « una hermosura de mujer » (0 vote, p = 0,72), « la mujer de mi vida » (0 vote), « this world wouldn't last a day without women ».
3. **Défense des femmes notée NO** par les annotateurs : « las mujeres no deben ser discriminadas salarialmente », « I hate when men rape and kill women ».

**Bugs de preprocessing trouvés en comparant `tweet` et `tweet_clean`**

| Bug | Exemple |
|---|---|
| la réduction des lettres répétées touche aussi les chiffres | « 1000 selfies » → « 100 selfies », « 200,000 years » → « 200,00 » |
| la regex d'URL mange le mot collé au lien | « https://t.co/jjeX7Y5magAnalicemos » → « url » (« Analicemos » perdu) |
| la regex de mention s'applique au milieu des mots | « Conectad@s » → « Conectad @user », « l@s » → « l @user » |
| les hashtags en minuscules ne sont pas découpés | #notallmen, #violenciamachista, #lamanada, #yositecreo |
| chaînes de mentions | jusqu'à 16 « @user » d'affilée (tokens perdus) |

**Le mot-clé de collecte.** Les identifiants du train et du dev sont rangés **par mot-clé de collecte** (blocs d'environ 20 tweets dans le train, 3 dans le dev : gamergate, harassed, MGTOW, notallmen, mansplaining…). La part de YES varie énormément d'un mot-clé à l'autre (mansplaining, superwoman syndrome, slutbag : presque tout YES ; violación « de la ley », conciliación, corresponsabilidad : presque tout NO). On reconstruit automatiquement ce lexique à partir du train, puis on retrouve le mot-clé d'un tweet **à partir de son seul texte** (sans utiliser l'identifiant). Sur le dev, ce prior a une corrélation de 0,57 avec la part de YES. C'est la feature du « premier modèle » la plus prometteuse.

**Hypothèses → expériences**

| Constat | Expérience |
|---|---|
| bugs de preprocessing | `prep_v2` (partie 6) |
| sous-confiance du soft, seuil | seuil choisi sur l'OOF (partout) |
| reported / judgemental confondus avec non sexiste | sorties **4 classes** et **multi-tâches** (le modèle voit l'intention) |
| ambiguïté (2 et 4 votes) | sortie **7 classes** (nombre de votes), soft labels |
| sujet du tweet (mot-clé) | prior du mot-clé, prior kNN, stacking, cascade (parties 4 et 10) |
| ironie, contexte, définition EXIST | **LLM 7B** : few-shot, justifications, fine-tuning LoRA (partie 9), routage des cas incertains |
| peu de données | **AEDA** |
| tête de classification | linéaire vs mean-pooling vs **MultiCNN** |

## 1. Installation, paramètres et chargement

In [1]:
%pip install -q "transformers>=4.51" "peft>=0.13" accelerate sentencepiece protobuf tqdm scikit-learn scipy

In [2]:
import json, re, html, math, time, gc, random, unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from scipy.sparse import hstack

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)
hf_logging.set_verbosity_error()

SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Ce notebook demande un GPU (Colab : Exécution > Modifier le type d'exécution > G4)"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(torch.cuda.get_device_name(0), "| précision :", AMP_DTYPE, "| mémoire :",
      round(torch.cuda.get_device_properties(0).total_memory / 1e9), "Go")

NVIDIA RTX PRO 6000 Blackwell Server Edition | précision : torch.bfloat16 | mémoire : 102 Go


**Paramètres d'exécution.** Tout est activé par défaut ; on peut couper une partie pour gagner du temps. Les résultats déjà en cache ne sont jamais recalculés.

In [3]:
N_FOLDS = 5                     # validation croisée sur le train
RUN_TRANSFORMERS = True         # partie 6 : baseline et ablations
RUN_COMBO = True                # partie 7 : meilleure combinaison, 3 seeds
RUN_LLM_PROMPT = True           # partie 8.1 : LLM few-shot
RUN_LLM_RATIONALES = True       # partie 8.2 : justifications générées + XLM-R
RUN_LLM_FT = True               # partie 8.3 : fine-tuning LoRA du LLM + distillation

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"   # utilisé pour les trois usages du LLM (7B)
# Alternatives de même taille : "Qwen/Qwen3-8B" (le mode « thinking » est désactivé), "meta-llama/Llama-3.1-8B-Instruct" (accès à demander)
LLM_FEWSHOT_K = 8               # nombre d'exemples dans le prompt (moitié YES, moitié NO)
EMBED_NAME = "intfloat/multilingual-e5-large"   # pour le kNN et le choix des exemples du few-shot
REFERENCE = "baseline"          # moyenne hard + soft de XLM-R large : la pipeline à battre

In [4]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/TweetClassification")
else:
    PROJECT_DIR = Path(".")

DATA_DIR = PROJECT_DIR
CACHE_DIR = PROJECT_DIR / "cache_modelling2"
(CACHE_DIR / "runs").mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = PROJECT_DIR / "dev_predictions_v2.csv"
RESULTS_CSV = PROJECT_DIR / "resultats_modelling2.csv"
print("Colab :", IN_COLAB, "| projet :", PROJECT_DIR.resolve(), "| cache :", CACHE_DIR)

Mounted at /content/drive
Colab : True | projet : /content/drive/MyDrive/TweetClassification | cache : /content/drive/MyDrive/TweetClassification/cache_modelling2


In [5]:
TASK2 = ["NO", "DIRECT", "REPORTED", "JUDGEMENTAL"]
TASK3 = ["IDEOLOGICAL-INEQUALITY", "STEREOTYPING-DOMINANCE", "OBJECTIFICATION",
         "SEXUAL-VIOLENCE", "MISOGYNY-NON-SEXUAL-VIOLENCE"]


def load_exist(name):
    path = next(p for p in DATA_DIR.rglob(name) if "cache" not in str(p))
    with open(path, encoding="utf-8") as f:
        df = pd.DataFrame.from_dict(json.load(f), orient="index").reset_index(drop=True)
    df["id"] = df["id_EXIST"].astype(int)
    return df.sort_values("id").reset_index(drop=True)   # l'ordre des id suit les mots-clés de collecte


def task2_dist(row):
    """Répartition des 6 votes entre NO / DIRECT / REPORTED / JUDGEMENTAL."""
    c = np.zeros(4)
    for l1, l2 in zip(row["labels_task1"], row["labels_task2"]):
        c[0 if l1 != "YES" else (TASK2.index(l2) if l2 in TASK2[1:] else 1)] += 1
    return c / c.sum()


def task3_frac(row):
    """Part des annotateurs qui citent chaque catégorie de la tâche 3 (multi-label)."""
    c = np.zeros(5)
    for cats in row["labels_task3"]:
        for cat in (cats if isinstance(cats, list) else [cats]):
            if cat in TASK3:
                c[TASK3.index(cat)] += 1
    return c / len(row["labels_task3"])


def add_labels(df):
    df = df.copy()
    df["n_ann"] = df["labels_task1"].str.len()
    df["n_yes"] = df["labels_task1"].apply(lambda l: sum(x == "YES" for x in l))
    df["p_yes"] = df["n_yes"] / df["n_ann"]
    df["hard"] = np.select([df["p_yes"] > 0.5, df["p_yes"] < 0.5], ["YES", "NO"], default=None)
    df["t2"] = df.apply(task2_dist, axis=1)
    df["t3"] = df.apply(task3_frac, axis=1)
    return df


train_df = add_labels(load_exist("EXIST2024_training.json"))
dev_df = add_labels(load_exist("EXIST2024_dev.json"))
N_TR, N_DEV = len(train_df), len(dev_df)

P_TR, P_DEV = train_df["p_yes"].to_numpy(), dev_df["p_yes"].to_numpy()
TIE_TR, TIE_DEV = train_df["hard"].isna().to_numpy(), dev_df["hard"].isna().to_numpy()
Y_TR = (train_df["hard"] == "YES").astype(int).to_numpy()     # 0 pour les égalités (jamais évaluées)
Y_DEV = (dev_df["hard"] == "YES").astype(int).to_numpy()
NYES_TR = train_df["n_yes"].to_numpy()
T2_TR, T3_TR = np.stack(train_df["t2"]), np.stack(train_df["t3"])
LANG_TR, LANG_DEV = train_df["lang"].to_numpy(), dev_df["lang"].to_numpy()

print(f"Train : {N_TR} tweets ({(~TIE_TR).sum()} avec étiquette hard) | Dev : {N_DEV} ({(~TIE_DEV).sum()})")
print("Répartition moyenne des intentions (train) :", dict(zip(TASK2, T2_TR.mean(0).round(3))))
print("Part moyenne des catégories (train) :", dict(zip(TASK3, T3_TR.mean(0).round(3))))

Train : 6920 tweets (6064 avec étiquette hard) | Dev : 1038 (934)
Répartition moyenne des intentions (train) : {'NO': np.float64(0.548), 'DIRECT': np.float64(0.219), 'REPORTED': np.float64(0.112), 'JUDGEMENTAL': np.float64(0.121)}
Part moyenne des catégories (train) : {'IDEOLOGICAL-INEQUALITY': np.float64(0.14), 'STEREOTYPING-DOMINANCE': np.float64(0.163), 'OBJECTIFICATION': np.float64(0.126), 'SEXUAL-VIOLENCE': np.float64(0.079), 'MISOGYNY-NON-SEXUAL-VIOLENCE': np.float64(0.099)}


Si le CSV du run précédent est sur Drive, on recalcule ses scores pour avoir la référence sous les yeux.

In [6]:
prev_csv = PROJECT_DIR / "dev_predictions.csv"
if prev_csv.exists():
    prev = pd.read_csv(prev_csv)
    h = prev[prev["hard"].notna()]
    y = (h["hard"] == "YES").astype(int)
    print("Run précédent (XLM-R large, 1 époque) :")
    for c in ["proba_hard", "proba_soft", "proba_ensemble"]:
        print(f"  {c:15s} F1@0,5 = {f1_score(y, h[c] >= .5):.4f}")
    display(pd.crosstab(h["n_yes"], h["error_type"].fillna("ok"), margins=True))
else:
    print("dev_predictions.csv absent : on passe.")

Run précédent (XLM-R large, 1 époque) :
  proba_hard      F1@0,5 = 0.8624
  proba_soft      F1@0,5 = 0.8650
  proba_ensemble  F1@0,5 = 0.8618


error_type,FN,FP,ok,All
n_yes,,,,
0,0,6,197,203
1,0,21,139,160
2,0,41,75,116
4,35,0,109,144
5,16,0,132,148
6,8,0,155,163
All,59,68,807,934


## 2. Preprocessing

Trois versions ; `light` (baseline) et `v2` sont comparées à réglages identiques en partie 6, `xlmt` sert d'entrée au LLM et aux embeddings :

- **`light`** : la version d'origine, **avec ses bugs**, pour reproduire le run précédent ;
- **`xlmt`** : le prétraitement officiel de Twitter-XLM-R (mentions → `@user`, liens → `http`, rien d'autre ; hashtags bruts). C'est le format vu pendant le pré-entraînement ;
- **`v2`** : `xlmt` + les corrections issues de l'analyse d'erreurs :
  - liens `t.co` coupés à leur vraie longueur (le mot collé est conservé) ;
  - mentions seulement en début de mot (`Conectad@s` reste intact) ;
  - chaînes de mentions ou de liens réduites à un seul jeton, préfixe « RT @x: » retiré ;
  - lettres répétées réduites, **mais pas les chiffres** ; rires normalisés (« jajajaja » → « jaja ») ;
  - hashtags découpés, **y compris en minuscules** grâce à un découpage par programmation dynamique sur le vocabulaire du train (#yositecreo → « yo si te creo »), en gardant le `#` (signal « campagne ») ;
  - phrases collées séparées (« capullo.El » → « capullo. El ») ;
  - casse, ponctuation (matière d'AEDA), emojis et stopwords conservés.

In [7]:
# --- version d'origine (bugs compris) ---
URL_OLD = re.compile(r"https?://\S+|www\.\S+")
MENTION_OLD = re.compile(r"@\w+")
HASHTAG = re.compile(r"#(\w+)")
CAMEL = re.compile(r"(?<=[a-záéíóúñü])(?=[A-ZÁÉÍÓÚÑÜ])")
REPEAT_OLD = re.compile(r"(\w)\1{2,}")


def prep_light(text):
    text = html.unescape(text)
    text = URL_OLD.sub(" url ", text)
    text = MENTION_OLD.sub(" @user ", text)
    text = HASHTAG.sub(lambda m: " " + CAMEL.sub(" ", m.group(1)) + " ", text)
    text = REPEAT_OLD.sub(r"\1\1", text)
    return " ".join(text.split())


# --- version officielle XLM-T, avec des regex correctes ---
TCO = re.compile(r"https?://t\.co/[A-Za-z0-9]{10}")
URL = re.compile(r"https?://\S+|www\.\S+")
MENTION = re.compile(r"(?<![\w@])@\w{1,15}")


def prep_xlmt(text):
    text = html.unescape(text)
    text = TCO.sub(" http ", text)
    text = URL.sub(" http ", text)
    text = MENTION.sub(" @user ", text)
    return " ".join(text.split())


# --- découpage des hashtags tout en minuscules ---
def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))


def build_vocab(texts, min_count=3):
    cnt = Counter()
    for t in texts:
        t = URL.sub(" ", html.unescape(t))
        t = re.sub(r"[@#]\w+", " ", t)
        cnt.update(re.findall(r"[a-z]+", strip_accents(t.lower())))
    extra = ("not all men yes all women me too ni una menos yo si te creo la manada women in tech womens day "
             "violencia machista everyday sexism stop street harassment womens rights we are equal "
             "international feminism feminismo feminista igualdad genero mujer mujeres")
    cnt.update({w: 50 for w in extra.split()})
    return {w: c for w, c in cnt.items() if c >= min_count and (len(w) > 1 or w in "aeoyu")}


VOCAB = build_vocab(train_df["tweet"])
VOCAB_TOTAL = sum(VOCAB.values())


def segment_word(word, max_piece=20):
    """Découpage par programmation dynamique (coût = -log fréquence + 1 par morceau)."""
    w = strip_accents(word.lower())
    n = len(w)
    best = [(0.0, [])] + [(math.inf, None)] * n
    for i in range(1, n + 1):
        for j in range(max(0, i - max_piece), i):
            f = VOCAB.get(w[j:i], 0)
            if f and best[j][0] < math.inf:
                cost = best[j][0] - math.log(f / VOCAB_TOTAL) + 1.0
                if cost < best[i][0]:
                    best[i] = (cost, best[j][1] + [w[j:i]])
    return best[n][1] if best[n][1] else [word]


def split_hashtag(tag):
    if re.search(r"[a-z][A-Z]|[A-Za-z][0-9]", tag):                 # CamelCase ou lettres+chiffres
        return re.sub(r"(?<=[a-záéíóúñü])(?=[A-ZÁÉÍÓÚÑÜ])|(?<=[A-Za-z])(?=[0-9])", " ", tag).split()
    if len(tag) >= 6 and (tag.islower() or tag.isupper()) and VOCAB.get(strip_accents(tag.lower()), 0) < 5:
        pieces = segment_word(tag)
        if len(pieces) > 1:
            return pieces
    return [tag]


REPEAT = re.compile(r"([^\W\d_])\1{2,}")
LAUGH = re.compile(r"\b([jh])([aeiou])(?:\1\2)+[jh]?\b|\b(?:js){2,}j?\b", re.I)   # jajaja, hahaha, jsjsjs (pas « hijo »)
GLUED = re.compile(r"(?<=[a-záéíóúñü])([.!?…])(?=[A-ZÁÉÍÓÚÑ¿¡])")


def prep_v2(text):
    text = html.unescape(text)
    text = re.sub(r"^RT\s+@?\w+:\s*", "", text)
    text = TCO.sub(" http ", text)
    text = URL.sub(" http ", text)
    text = MENTION.sub(" @user ", text)
    text = re.sub(r"(?:@user\s*){2,}", "@user ", text)
    text = re.sub(r"(?:http\s*){2,}", "http ", text)
    text = HASHTAG.sub(lambda m: " #" + " ".join(split_hashtag(m.group(1))) + " ", text)
    text = LAUGH.sub(lambda m: m.group(0)[:2] * 2, text)
    text = REPEAT.sub(r"\1\1", text)
    text = GLUED.sub(r"\1 ", text)
    return " ".join(text.split())


PREPS = {"light": prep_light, "xlmt": prep_xlmt, "v2": prep_v2}
TEXTS = {name: (train_df["tweet"].map(f).to_numpy(), dev_df["tweet"].map(f).to_numpy())
         for name, f in PREPS.items()}

examples = [s for s in train_df["tweet"] if re.search(r"#[a-z]{8,}\b|t\.co/\w{10}[A-Za-z]|\w@\w|\d{4,}|@\w+\s+@\w+\s+@\w+", s)][:8]
display(pd.DataFrame({name: [f(s) for s in examples] for name, f in PREPS.items()}, index=[s[:60] for s in examples]))
print([(h, split_hashtag(h)) for h in ["notallmen", "yositecreo", "violenciamachista", "lamanada", "womenintech", "IDoNotRenounce"]])

,light,xlmt,v2
"@Steven2897 Lee sobre Gamergate, y como eso ha cambiado la m","@user Lee sobre Gamergate, y como eso ha cambiado la manera en la cual nos comunicamos en el internet. Los fanboys de Halo están tóxicos...","@user Lee sobre Gamergate, y como eso ha cambiado la manera en la cual nos comunicamos en el internet. Los fanboys de Halo están tóxicos...","@user Lee sobre Gamergate, y como eso ha cambiado la manera en la cual nos comunicamos en el internet. Los fanboys de Halo están tóxicos..."
@novadragon21 @icep4ck @TvDannyZ Entonces como así es el mer,@user @user @user Entonces como así es el mercado lo mejor no es hacer algo para cambiarlo y seguir alimentando el machismo en los consu...,@user @user @user Entonces como así es el mercado lo mejor no es hacer algo para cambiarlo y seguir alimentando el machismo en los consu...,@user Entonces como así es el mercado lo mejor no es hacer algo para cambiarlo y seguir alimentando el machismo en los consumidores en l...
.¿Conoces la #DECORACION #estilo #GAMER para #adolescentes,.¿Conoces la DECORACION estilo GAMER para adolescentes ?Te lo explicamos en nuestro post : ❤️ url sigues? inspiracionalcuadrado interior...,.¿Conoces la #DECORACION #estilo #GAMER para #adolescentes ?Te lo explicamos en nuestro #post : ❤️ http ¿Nos sigues?#inspiracionalcuadra...,.¿Conoces la #DECORACION #estilo #GAMER para #adolescentes ?Te lo explicamos en nuestro #post : ❤️ http ¿Nos sigues? #inspiracionalcuadr...
CES 2022 ASUS ROG Rise of Gamers Evento de lanzamiento https,CES 2022 ASUS ROG Rise of Gamers Evento de lanzamiento url AMD2022 AMD AMDRed Team AMDRyzen Intel nvidia NVIDIAOmniverse NVIDIAGe Force ...,CES 2022 ASUS ROG Rise of Gamers Evento de lanzamiento http #AMD2022 #AMD #AMDRedTeam #AMDRyzen #Intel #nvidia #NVIDIAOmniverse #NVIDIAG...,CES 2022 ASUS ROG Rise of Gamers Evento de lanzamiento http #AMD 2022 #AMD #AMDRed Team #AMDRyzen #Intel #nvidia #NVIDIAOmniverse #NVIDI...
¿Y por qué digo que son dogwhistles? Porque esto ya lo hemos,"¿Y por qué digo que son dogwhistles? Porque esto ya lo hemos vivido una y otra y otra vez en los cómics... Pasó con Comics Gate en 2017,...","¿Y por qué digo que son dogwhistles? Porque esto ya lo hemos vivido una y otra y otra vez en los cómics... Pasó con #ComicsGate en 2017,...",¿Y por qué digo que son dogwhistles? Porque esto ya lo hemos vivido una y otra y otra vez en los cómics... Pasó con #Comics Gate en 2017...
La que se levanto sin miedo al exito fue la rata Nro 1 de Ru,La que se levanto sin miedo al exito fue la rata Nro 1 de Rust.... @user url gamergirls GAMERGATE url,La que se levanto sin miedo al exito fue la rata Nro 1 de Rust.... @user http #Rust #gamergirls #GAMERGATE http,La que se levanto sin miedo al exito fue la rata Nro 1 de Rust.... @user http #Rust #gamer girls #GAMERGATE http
@BrunoLouviers en 2014 empezó gamergate así que igual mejor,@user en 2014 empezó gamergate así que igual mejor no,@user en 2014 empezó gamergate así que igual mejor no,@user en 2014 empezó gamergate así que igual mejor no
Los mejores 100 #videojuegos de la #Historia 😀😀😀😀 #gamers #G,Los mejores 100 videojuegos de la Historia 😀😀😀😀 gamers Gamer Gate url,Los mejores 100 #videojuegos de la #Historia 😀😀😀😀 #gamers #GamerGate http,Los mejores 100 #video juegos de la #Historia 😀😀😀😀 #gamers #Gamer Gate http


[('notallmen', ['notallmen']), ('yositecreo', ['yo', 'site', 'creo']), ('violenciamachista', ['violencia', 'machista']), ('lamanada', ['lamanada']), ('womenintech', ['women', 'in', 'tech']), ('IDoNotRenounce', ['IDo', 'Not', 'Renounce'])]


## 3. Protocole d'évaluation

- **5 folds stratifiés** (langue × nombre de votes YES), seed 42, partagés par **tous** les modèles : les prédictions OOF sont donc comparables et empilables.
- **Mesure principale : F1 de la classe YES**, sur les tweets sans égalité.
- **Seuil** : 0,5 et seuil optimal **lu sur l'OOF**, puis appliqué tel quel au dev.
- **Incertitude** : intervalle de confiance bootstrap à 95 % sur le dev, et **test bootstrap apparié** contre la baseline (part des rééchantillonnages où l'écart est ≤ 0).
- Chaque composant (modèle, feature, LLM, combinaison) est enregistré dans un registre commun `REG` : `oof` (6 920 probabilités) et `dev` (1 038 probabilités).

In [8]:
strat = train_df["lang"] + "_" + train_df["n_yes"].astype(str)
FOLDS = np.zeros(N_TR, dtype=int)
for k, (_, va) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(N_TR), strat)):
    FOLDS[va] = k

EVAL_TR, EVAL_DEV = ~TIE_TR, ~TIE_DEV
THR_GRID = np.round(np.arange(0.20, 0.801, 0.01), 2)
REG = {}          # nom -> {"oof", "dev", "kind", ...}


def clip(p):
    return np.clip(np.asarray(p, dtype=float), 1e-4, 1 - 1e-4)


def logit(p):
    p = clip(p)
    return np.log(p / (1 - p))


def f1_at(y, p, thr=0.5):
    return f1_score(y, (p >= thr).astype(int), zero_division=0)


def best_threshold(p_oof):
    m = EVAL_TR & ~np.isnan(p_oof)
    scores = [f1_at(Y_TR[m], p_oof[m], t) for t in THR_GRID]
    i = int(np.argmax(scores))
    return float(THR_GRID[i]), scores[i]


def soft_ce(p_true, p_pred):
    m = ~np.isnan(p_pred)
    p = clip(p_pred[m])
    return float(-np.mean(p_true[m] * np.log(p) + (1 - p_true[m]) * np.log(1 - p)))


RNG_BOOT = np.random.default_rng(0)
BOOT_IDX = RNG_BOOT.integers(0, EVAL_DEV.sum(), size=(2000, EVAL_DEV.sum()))


def boot_f1(pred):
    """F1 vectorisé sur 2 000 rééchantillonnages du dev (mêmes tirages pour tous les modèles)."""
    y = Y_DEV[EVAL_DEV][BOOT_IDX]
    p = pred[BOOT_IDX]
    tp = (y * p).sum(1); fp = ((1 - y) * p).sum(1); fn = (y * (1 - p)).sum(1)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)


def register(name, oof, dev, kind, **info):
    REG[name] = dict(oof=np.asarray(oof, dtype=float), dev=np.asarray(dev, dtype=float), kind=kind, **info)


def oof_score(name):
    return best_threshold(REG[name]["oof"])[1]


def evaluate(name, ref=None):
    ref = ref or REFERENCE
    r = REG[name]
    thr, f1_oof_best = best_threshold(r["oof"])
    pred_dev = (r["dev"][EVAL_DEV] >= thr).astype(int)
    bs = boot_f1(pred_dev)
    row = {
        "nom": name, "type": r["kind"],
        "OOF F1@0.5": f1_at(Y_TR[EVAL_TR & ~np.isnan(r["oof"])], r["oof"][EVAL_TR & ~np.isnan(r["oof"])]),
        "seuil OOF": thr, "OOF F1@seuil": f1_oof_best,
        "OOF CE soft": soft_ce(P_TR, r["oof"]),
        "dev F1@0.5": f1_at(Y_DEV[EVAL_DEV], r["dev"][EVAL_DEV]),
        "dev F1@seuil": f1_at(Y_DEV[EVAL_DEV], r["dev"][EVAL_DEV], thr),
        "IC95 bas": np.percentile(bs, 2.5), "IC95 haut": np.percentile(bs, 97.5),
        "dev EN": f1_at(Y_DEV[EVAL_DEV & (LANG_DEV == "en")], r["dev"][EVAL_DEV & (LANG_DEV == "en")], thr),
        "dev ES": f1_at(Y_DEV[EVAL_DEV & (LANG_DEV == "es")], r["dev"][EVAL_DEV & (LANG_DEV == "es")], thr),
        "dev CE soft": soft_ce(P_DEV, r["dev"]),
    }
    if ref in REG and ref != name:
        thr_ref, _ = best_threshold(REG[ref]["oof"])
        bs_ref = boot_f1((REG[ref]["dev"][EVAL_DEV] >= thr_ref).astype(int))
        row["Δ dev vs réf"] = row["dev F1@seuil"] - f1_at(Y_DEV[EVAL_DEV], REG[ref]["dev"][EVAL_DEV], thr_ref)
        row["p(Δ≤0)"] = float(np.mean(bs - bs_ref <= 0))
    return row


def summary(names=None, sort="OOF F1@seuil", ref=None):
    names = list(dict.fromkeys(names or REG))
    df = pd.DataFrame([evaluate(n, ref) for n in names]).set_index("nom")
    return df.sort_values(sort, ascending=False).round(4)

## 4. Le « premier modèle » : signaux bon marché

Idée : un premier modèle rapide classe une partie des tweets presque sûrement, et tous les tweets passent quand même par BERT ; on combine ensuite les deux (partie 10). On construit ici les briques, toutes en OOF sur les mêmes folds :

1. **TF-IDF mots + caractères, régression logistique soft** ;
2. **25 features manuelles** (lexiques bilingues construits d'après le rapport et l'analyse d'erreurs, sans regarder le dev) ;
3. **prior du mot-clé de collecte** : lexique reconstruit à partir des blocs d'identifiants du train, puis encodage de la cible lissé, en OOF ;
4. **prior kNN** : part moyenne de YES des 25 tweets du train les plus proches (embeddings `multilingual-e5-large`) ; les mêmes embeddings servent à choisir les exemples du few-shot ;
5. **règle lexique** : « mot du lexique sexiste ⇒ YES ».

### 4.1 Texte normalisé et features manuelles

In [9]:
def norm_text(t):
    """Minuscules, sans accents, hashtags découpés, ponctuation retirée : pour TF-IDF et les lexiques."""
    t = html.unescape(t)
    t = TCO.sub(" http ", t); t = URL.sub(" http ", t)
    t = MENTION.sub(" atuser ", t)
    t = HASHTAG.sub(lambda m: " " + " ".join(split_hashtag(m.group(1))) + " " + m.group(1) + " ", t)
    t = strip_accents(t.lower())
    t = REPEAT.sub(r"\1\1", t)
    return " ".join(re.sub(r"[^a-z0-9ñ ]+", " ", t).split())


NORM_TR = train_df["tweet"].map(norm_text).to_numpy()
NORM_DEV = dev_df["tweet"].map(norm_text).to_numpy()


def wb(words):
    return re.compile(r"\b(?:" + "|".join(words) + r")\b")


LEXICONS = {   # appliqués au texte normalisé
    "lex_sexist": wb([r"feminazis?", r"whores?", r"sluts?", r"slutbag\w*", r"skanks?", r"cunts?", r"zorras?", r"guarras?",
                      r"golfas?", r"marimachos?", r"sumisas?", r"dumb blondes?", r"rubias? tontas?", r"hembrismo", r"hembrista\w*",
                      r"mujer florero", r"gold diggers?", r"make me a sandwich", r"frigidas?", r"lagartonas?", r"nenazas?",
                      r"machorras?", r"furcias?", r"fulanas?", r"rameras?", r"thots?", r"femoids?", r"foids?"]),
    "generalisation": re.compile(r"\b(?:women|woman|girls|females|ladies|wives|mujeres|una mujer|las chicas|las ninas|"
                                 r"las senoritas|feministas|feminists)\s+(?:are|r|can t|cannot|don t|do not|shouldn t|should|"
                                 r"always|never|need|just|only|belong|son|estan|no saben|no pueden|no deben|no deberian|no sirven|"
                                 r"siempre|nunca|solo|deben|deberian|necesitan)\b"),
    "trad_role": wb([r"kitchen", r"sandwich\w*", r"cook for", r"dishes", r"laundry", r"housewife", r"cocina\w*", r"fregar",
                     r"fregona", r"lavar (?:los platos|la ropa)", r"ama de casa", r"a la casa", r"planchar"]),
    "feminism_vocab": wb([r"feminis\w*", r"feminazi\w*", r"hembris\w*", r"sororidad", r"patriarca\w*", r"patriarchy",
                          r"patriarchal", r"8m", r"ni una menos", r"niunamenos", r"empoderad\w*"]),
    "sexism_vocab": wb([r"sexis\w*", r"misogin\w*", r"misogyn\w*", r"machis\w*", r"micromachis\w*", r"mansplain\w*",
                        r"manspread\w*", r"acos\w*", r"harass\w*", r"violaci\w*", r"violad\w*", r"rape\w*", r"rapist\w*",
                        r"abus\w*", r"maltrat\w*", r"brecha", r"glass ceiling", r"techo de cristal", r"desigualdad",
                        r"inequality", r"gender gap", r"pay gap", r"feminicid\w*", r"femicid\w*", r"violencia de genero",
                        r"violencia machista", r"cosifica\w*", r"objectif\w*", r"consent\w*", r"discrimina\w*"]),
    "gender_insult": wb([r"bitch\w*", r"whore\w*", r"slut\w*", r"cunt\w*", r"hoes?", r"thots?", r"skank\w*", r"putas?",
                         r"zorras?", r"perras?", r"guarras?", r"golfas?", r"furcias?", r"pendeja\w*", r"tontas?", r"gordas?",
                         r"feas?"]),
    "women_ref": wb([r"women", r"woman", r"girls?", r"females?", r"she", r"her", r"hers", r"lady", r"ladies", r"wife",
                     r"wives", r"mom", r"mother", r"mujer\w*", r"chicas?", r"ninas?", r"senoras?", r"senoritas?",
                     r"esposas?", r"novias?", r"ellas?", r"madres?", r"morras?", r"minas?"]),
    "men_ref": wb([r"men", r"man", r"guys?", r"males?", r"boys?", r"he", r"him", r"his", r"husband", r"hombres?",
                   r"chicos?", r"tios?", r"varon\w*", r"machos?", r"maridos?", r"novios?", r"ellos"]),
    "body_sex": wb([r"tits", r"boobs", r"ass", r"pussy", r"dick", r"cock", r"cum", r"sexy", r"nudes?", r"culo\w*",
                    r"tetas?", r"chichis", r"nalgas", r"follar\w*", r"coger", r"pollas?", r"vergas?", r"vagina\w*",
                    r"panocha", r"cuerpo\w*", r"body", r"bikini", r"faldas?", r"skirts?", r"sujetador", r"bragas"]),
    "personal_exp": re.compile(r"\b(?:me (?:dijo|dijeron|grito|gritaron|toco|tocaron|siguio|siguieron|acoso|acosaron|"
                               r"llamo|llamaron)|i (?:was|got|ve been|have been) (?:harassed|raped|assaulted|catcalled|groped|"
                               r"followed|called)|sufri\w*|my (?:boss|ex|husband|boyfriend)|mi (?:jefe|ex|marido|novio))\b"),
    "negation": wb([r"not", r"no", r"never", r"nunca", r"jamas", r"ni", r"don t", r"doesn t", r"isn t"]),
    "second_person": wb([r"you", r"your", r"u", r"ur", r"tu", r"te", r"vos", r"usted", r"ustedes", r"tus", r"ti"]),
    "first_person": wb([r"i", r"me", r"my", r"yo", r"mi", r"mis"]),
    "laugh_word": wb([r"jaja\w*", r"haha\w*", r"lol", r"lmao", r"jsjs\w*", r"xd+"]),
}
RAW_PATTERNS = {   # appliqués au texte brut
    "denunciation_hashtag": re.compile(r"#(?:metoo|yositecreo|niunamenos|8m|cuentalo|everydaysexism|stopstreetharassment|"
                                       r"violenciamachista|timesup|yesallwomen|notallmen)\b", re.I),
    "campaign_hashtag": re.compile(r"#(?:womensday|internationalwomensday|iwd\w*|diadelamujer\w*|diainternacionaldelamujer|"
                                   r"womenintech|womenempowerment|weareequal|womensrights|empoderamiento\w*|"
                                   r"womensupportingwomen|girlpower)\b", re.I),
    "has_hashtag": re.compile(r"#\w+"),
    "has_url": re.compile(r"https?://"),
    "n_mentions": re.compile(r"(?<![\w@])@\w+"),
    "question": re.compile(r"\?"),
    "quotes": re.compile(r"[\"“”«»]"),
    "laugh_emoji": re.compile(r"[😂🤣]"),
    "exasperation_emoji": re.compile(r"[🙄🤦🤷😒😤]"),
    "heart_emoji": re.compile(r"[❤💕💖💗💜💚💙🥰😍😘]"),
}
FEATURE_NAMES = list(LEXICONS) + list(RAW_PATTERNS) + ["is_reply", "log_len"]


def hand_features(raw, normed):
    rows = []
    for t, n in zip(raw, normed):
        r = [min(len(p.findall(n)), 3) for p in LEXICONS.values()]
        r += [min(len(p.findall(t)), 3) for p in RAW_PATTERNS.values()]
        r += [float(t.lstrip().startswith("@")), math.log1p(len(n.split()))]
        rows.append(r)
    return np.array(rows, dtype=np.float32)


HF_TR = hand_features(train_df["tweet"], NORM_TR)
HF_DEV = hand_features(dev_df["tweet"], NORM_DEV)
_mu, _sd = HF_TR.mean(0), HF_TR.std(0) + 1e-6
HFZ_TR, HFZ_DEV = (HF_TR - _mu) / _sd, (HF_DEV - _mu) / _sd   # centrées-réduites (stats du train)

# Part de YES quand la feature est présente (train, hors égalités) : vérifie que chaque feature porte un signal
stats = []
for j, name in enumerate(FEATURE_NAMES[:-1]):
    present = (HF_TR[:, j] > 0) & EVAL_TR
    stats.append({"feature": name, "part du train": present.mean(), "part de YES si présente": Y_TR[present].mean() if present.any() else np.nan})
display(pd.DataFrame(stats).set_index("feature").round(3).sort_values("part de YES si présente", ascending=False))

,part du train,part de YES si présente
feature,,
generalisation,0.042,0.873
lex_sexist,0.061,0.870
trad_role,0.005,0.816
gender_insult,0.063,0.747
feminism_vocab,0.064,0.733
women_ref,0.301,0.712
body_sex,0.062,0.641
sexism_vocab,0.141,0.631
exasperation_emoji,0.016,0.615


### 4.2 TF-IDF (OOF) et features manuelles seules

In [10]:
def soft_logreg_fit(X, p, C):
    """Régression logistique sur cibles soft : chaque tweet dupliqué (YES poids p, NO poids 1-p)."""
    from scipy.sparse import vstack, issparse
    Xd = vstack([X, X]) if issparse(X) else np.vstack([X, X])
    yd = np.r_[np.ones(X.shape[0]), np.zeros(X.shape[0])]
    w = np.r_[p, 1 - p]
    return LogisticRegression(C=C, max_iter=3000).fit(Xd, yd, sample_weight=w)


def oof_and_dev(make_X, C):
    """make_X(train_idx) -> (X_train, transform) ; renvoie les probabilités OOF et dev (modèle refit sur tout le train)."""
    oof = np.zeros(N_TR)
    for k in range(N_FOLDS):
        tr, va = FOLDS != k, FOLDS == k
        Xtr, transform = make_X(tr)
        oof[va] = soft_logreg_fit(Xtr, P_TR[tr], C).predict_proba(transform("tr", va))[:, 1]
    Xall, transform = make_X(np.ones(N_TR, bool))
    dev = soft_logreg_fit(Xall, P_TR, C).predict_proba(transform("dev", None))[:, 1]
    return oof, dev


def make_tfidf(mask):
    w = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(NORM_TR[mask])
    c = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=200_000).fit(NORM_TR[mask])
    tf = lambda texts: hstack([w.transform(texts), c.transform(texts)]).tocsr()
    return tf(NORM_TR[mask]), lambda split, idx: tf(NORM_TR[idx] if split == "tr" else NORM_DEV)


oof, dev = oof_and_dev(make_tfidf, C=2.0)
register("tfidf", oof, dev, "modèle 1")



def make_hand(mask):
    return HFZ_TR[mask], lambda split, idx: HFZ_TR[idx] if split == "tr" else HFZ_DEV


oof, dev = oof_and_dev(make_hand, C=1.0)
register("features_manuelles", oof, dev, "modèle 1")
summary(["tfidf", "features_manuelles"])

,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft
nom,,,,,,,,,,,,
tfidf,modèle 1,0.7468,0.44,0.7743,0.5777,0.7665,0.7894,0.7582,0.8171,0.7766,0.7985,0.5716
features_manuelles,modèle 1,0.6809,0.38,0.7549,0.6105,0.7217,0.7825,0.7520,0.8088,0.7650,0.7964,0.6026


### 4.3 Prior du mot-clé de collecte

Le train est rangé par mot-clé. Pour chaque tweet du train, on cherche le n-gramme (1 à 4 mots) partagé par le plus de voisins de même langue dans une fenêtre de ±12 identifiants : c'est son mot-clé. On garde les mots-clés vus au moins 5 fois → un lexique d'environ 350 expressions. Ensuite, **pour n'importe quel tweet (dev compris), le mot-clé est retrouvé à partir du texte seul** : l'expression du lexique la plus longue qu'il contient. Le prior est la part moyenne de YES des tweets du train ayant ce mot-clé, lissée vers la moyenne globale (k = 10), calculée en OOF.

In [11]:
def ngram_set(normed, nmax=4):
    toks = normed.split()
    return {" ".join(toks[i:i + n]) for n in range(1, nmax + 1) for i in range(len(toks) - n + 1)}


NG_TR = [ngram_set(t) for t in NORM_TR]
NG_DEV = [ngram_set(t) for t in NORM_DEV]
DOC_FREQ = Counter(g for s in NG_TR for g in s)

seed_cache = CACHE_DIR / "seed_keywords.json"
if seed_cache.exists():
    block_seed = json.load(open(seed_cache, encoding="utf-8"))
else:
    W = 12
    block_seed = []
    for i in tqdm(range(N_TR), desc="mots-clés"):
        win = [j for j in range(max(0, i - W), min(N_TR, i + W + 1)) if j != i and LANG_TR[j] == LANG_TR[i]]
        best, best_score = None, 0.0
        for g in NG_TR[i]:
            if len(g) < 3 or DOC_FREQ[g] > 400 or g in {"atuser", "http"}:
                continue
            c = sum(g in NG_TR[j] for j in win)
            score = c + 0.3 * len(g.split())
            if c >= 4 and score > best_score:
                best, best_score = g, score
        block_seed.append(best)
    json.dump(block_seed, open(seed_cache, "w", encoding="utf-8"))

seed_counts = Counter(s for s in block_seed if s)
SEED_LEX = {s: c for s, c in seed_counts.items() if c >= 5}


def match_seed(ngrams):
    cands = [g for g in ngrams if g in SEED_LEX]
    return max(cands, key=lambda g: (len(g.split()), SEED_LEX[g])) if cands else None


SEED_TR = np.array([match_seed(s) for s in NG_TR], dtype=object)
SEED_DEV = np.array([match_seed(s) for s in NG_DEV], dtype=object)
print(f"Lexique : {len(SEED_LEX)} mots-clés | couverture train {np.mean(SEED_TR != None):.1%}, dev {np.mean(SEED_DEV != None):.1%}")


def target_encode(keys_fit, p_fit, keys_apply, k=10):
    prior = p_fit.mean()
    s = pd.DataFrame({"key": keys_fit, "p": p_fit}).dropna().groupby("key")["p"].agg(["sum", "count"])
    enc = ((s["sum"] + k * prior) / (s["count"] + k)).to_dict()
    return np.array([enc.get(key, prior) for key in keys_apply])


oof = np.zeros(N_TR)
for k in range(N_FOLDS):
    tr, va = FOLDS != k, FOLDS == k
    oof[va] = target_encode(SEED_TR[tr], P_TR[tr], SEED_TR[va])
register("prior_mot_cle", oof, target_encode(SEED_TR, P_TR, SEED_DEV), "modèle 1")

kw = pd.DataFrame({"mot-clé": SEED_TR, "p": P_TR}).dropna().groupby("mot-clé")["p"].agg(["mean", "count"])
kw = kw[kw["count"] >= 10].sort_values("mean")
print("Mots-clés les moins sexistes :", list(kw.index[:12]))
print("Mots-clés les plus sexistes :", list(kw.index[-12:]))
print("Corrélation prior / part de YES — OOF :", np.corrcoef(oof, P_TR)[0, 1].round(3),
      "| dev :", np.corrcoef(REG["prior_mot_cle"]["dev"], P_DEV)[0, 1].round(3))
summary(["tfidf", "features_manuelles", "prior_mot_cle"])

Lexique : 355 mots-clés | couverture train 98.0%, dev 97.2%
Mots-clés les moins sexistes : ['conciliar', 'estrecha', 'doble jornada', 'ecofeminismo', 'pnd', 'corresponsabilidad', 'marginalize', 'repression', 'metanarrative', 'essentialism', 'canonization', 'deserve']
Mots-clés les plus sexistes : ['bien como un hombre', 'feminazi', 'marimacho', 'cosificacion del cuerpo femenino', 'rubia tonta', 'entender a las mujeres', 'mujer al volante', 'las mujeres no', 'mucho feminismo pero', 'de hoy en dia', 'dumb blonde', 'i hate women']
Corrélation prior / part de YES — OOF : 0.591 | dev : 0.573


,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft
nom,,,,,,,,,,,,
tfidf,modèle 1,0.7468,0.44,0.7743,0.5777,0.7665,0.7894,0.7582,0.8171,0.7766,0.7985,0.5716
features_manuelles,modèle 1,0.6809,0.38,0.7549,0.6105,0.7217,0.7825,0.7520,0.8088,0.7650,0.7964,0.6026
prior_mot_cle,modèle 1,0.7091,0.46,0.7374,0.6106,0.7035,0.7291,0.6942,0.7609,0.7179,0.7375,0.6117


### 4.4 Embeddings, prior kNN

`multilingual-e5-large` (préfixe `query:`), moyenne des tokens, vecteurs normalisés. Pour un tweet du train, les voisins sont cherchés **dans les autres folds** et les quasi-doublons (cosinus > 0,98) sont écartés, pour imiter la situation du dev.

In [12]:
def embed_texts(texts, name=EMBED_NAME, bs=128):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModel.from_pretrained(name, torch_dtype=torch.float16).to(device).eval()
    order = np.argsort([len(t) for t in texts])
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)
    with torch.no_grad():
        for s in tqdm(range(0, len(texts), bs), desc="embeddings"):
            idx = order[s:s + bs]
            enc = tok(["query: " + texts[i] for i in idx], truncation=True, max_length=256, padding=True, return_tensors="pt").to(device)
            h = model(**enc).last_hidden_state
            m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
            e = (h * m).sum(1) / m.sum(1)
            out[idx] = F.normalize(e.float(), dim=-1).cpu().numpy()
    del model; gc.collect(); torch.cuda.empty_cache()
    return out


emb_cache = CACHE_DIR / "embeddings_e5.npz"
if emb_cache.exists():
    z = np.load(emb_cache); EMB_TR, EMB_DEV = z["tr"], z["dev"]
else:
    EMB_TR = embed_texts(list(TEXTS["xlmt"][0])); EMB_DEV = embed_texts(list(TEXTS["xlmt"][1]))
    np.savez(emb_cache, tr=EMB_TR, dev=EMB_DEV)

SIM_TR = EMB_TR @ EMB_TR.T          # 6 920 × 6 920
SIM_DEV = EMB_DEV @ EMB_TR.T        # 1 038 × 6 920


def allowed_neighbours(i_query=None):
    """Masque des tweets du train utilisables comme voisins (autres folds, pas de quasi-doublon)."""
    if i_query is None:
        return np.ones(N_TR, bool)
    return (FOLDS != FOLDS[i_query]) & (SIM_TR[i_query] < 0.98)


def knn_prior(sims, allowed, k=25, temp=0.05):
    s = np.where(allowed, sims, -np.inf)
    top = np.argpartition(-s, k)[:k]
    w = np.exp((s[top] - s[top].max()) / temp)
    return float((w * P_TR[top]).sum() / w.sum())


oof = np.array([knn_prior(SIM_TR[i], allowed_neighbours(i)) for i in range(N_TR)])
dev = np.array([knn_prior(SIM_DEV[i], allowed_neighbours()) for i in range(N_DEV)])
register("prior_knn", oof, dev, "modèle 1")
summary(["tfidf", "features_manuelles", "prior_mot_cle", "prior_knn"])

,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft
nom,,,,,,,,,,,,
prior_knn,modèle 1,0.7895,0.57,0.8035,0.5967,0.8176,0.8107,0.7818,0.8370,0.7737,0.8378,0.5742
tfidf,modèle 1,0.7468,0.44,0.7743,0.5777,0.7665,0.7894,0.7582,0.8171,0.7766,0.7985,0.5716
features_manuelles,modèle 1,0.6809,0.38,0.7549,0.6105,0.7217,0.7825,0.7520,0.8088,0.7650,0.7964,0.6026
prior_mot_cle,modèle 1,0.7091,0.46,0.7374,0.6106,0.7035,0.7291,0.6942,0.7609,0.7179,0.7375,0.6117


### 4.5 Le premier modèle complet

Une régression logistique sur [TF-IDF, prior mot-clé, prior kNN, features manuelles], entraînée sur les prédictions OOF (stacking). C'est le « modèle 1 » de la cascade testée en partie 10. Le stacking est lui-même évalué en validation croisée imbriquée (les mêmes folds), donc son OOF reste honnête.

In [13]:
def stack_matrix(names, with_hand=False, with_lang=True):
    cols_tr = [logit(REG[n]["oof"]) for n in names]
    cols_dev = [logit(REG[n]["dev"]) for n in names]
    if with_lang:
        cols_tr.append((LANG_TR == "en").astype(float)); cols_dev.append((LANG_DEV == "en").astype(float))
    Xtr, Xdev = np.column_stack(cols_tr), np.column_stack(cols_dev)
    if with_hand:
        Xtr, Xdev = np.hstack([Xtr, HFZ_TR]), np.hstack([Xdev, HFZ_DEV])
    return Xtr, Xdev


def fit_stack(name, names, with_hand=False, model="lr", C=0.5):
    Xtr, Xdev = stack_matrix(names, with_hand)
    make = (lambda: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))) if model == "lr" else \
           (lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0, random_state=SEED))
    m = EVAL_TR
    oof = np.full(N_TR, np.nan)
    for k in range(N_FOLDS):
        tr, va = m & (FOLDS != k), FOLDS == k
        oof[va] = make().fit(Xtr[tr], Y_TR[tr]).predict_proba(Xtr[va])[:, 1]
    dev = make().fit(Xtr[m], Y_TR[m]).predict_proba(Xdev)[:, 1]
    register(name, oof, dev, "combinaison", inputs=names, hand=with_hand, meta=model)


MODEL1_INPUTS = ["tfidf", "prior_mot_cle", "prior_knn"]
fit_stack("modele1", MODEL1_INPUTS, with_hand=True)
summary(["tfidf", "features_manuelles", "prior_mot_cle", "prior_knn", "modele1"])

,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft
nom,,,,,,,,,,,,
modele1,combinaison,0.8250,0.35,0.8329,0.6107,0.8288,0.8413,0.8154,0.8659,0.8235,0.8540,0.5974
prior_knn,modèle 1,0.7895,0.57,0.8035,0.5967,0.8176,0.8107,0.7818,0.8370,0.7737,0.8378,0.5742
tfidf,modèle 1,0.7468,0.44,0.7743,0.5777,0.7665,0.7894,0.7582,0.8171,0.7766,0.7985,0.5716
features_manuelles,modèle 1,0.6809,0.38,0.7549,0.6105,0.7217,0.7825,0.7520,0.8088,0.7650,0.7964,0.6026
prior_mot_cle,modèle 1,0.7091,0.46,0.7374,0.6106,0.7035,0.7291,0.6942,0.7609,0.7179,0.7375,0.6117


## 5. Fine-tuning : le cadre commun

Une seule classe `TweetClassifier` couvre toutes les variantes :

- **encodeur** : n'importe quel modèle Hugging Face (`AutoModel`) ;
- **tête** : `cls` (dense + tanh sur le premier token, comme RoBERTa), `mean` (moyenne des tokens), `cnn` (MultiCNN : convolutions de largeurs 4, 6, 8, max-pooling masqué, dense + ReLU) ;
- **cible** :
  - `hard` : binaire, vote majoritaire, égalités retirées de l'entraînement ;
  - `soft` : binaire, cible = part de YES ;
  - `cls4` : **4 classes** NO / DIRECT / REPORTED / JUDGEMENTAL, cible = répartition des 6 votes ; P(YES) = 1 − P(NO) ;
  - `cls7` : **7 classes** = nombre de votes YES (0 à 6) ; P(YES) = espérance du nombre de votes / 6 ;
  - `multitask` : binaire soft + têtes auxiliaires pour la tâche 2 (4 intentions) et la tâche 3 (5 catégories, multi-label), poids 0,5 ;
  - `distill` : binaire, cible = moyenne de la part de YES et de la probabilité OOF du LLM fine-tuné (partie 9.3) ;
- **AEDA** : à chaque époque, une copie de chaque tweet avec 1 à n/3 signes de ponctuation insérés au hasard ;
- **optimisation** : AdamW (fusionné), warmup 10 % puis décroissance linéaire, calcul en bf16, gradient clipping.

À chaque époque, on prédit le fold de validation **et** le dev : on choisit ensuite l'époque sur l'OOF (même époque pour les 5 folds), sans réentraîner.

In [14]:
TOKENIZERS = {}


def get_tokenizer(name):
    if name not in TOKENIZERS:
        TOKENIZERS[name] = AutoTokenizer.from_pretrained(name)
    return TOKENIZERS[name]


N_OUT = {"hard": 2, "soft": 2, "multitask": 2, "distill": 2, "cls4": 4, "cls7": 7}
SOFT_TARGETS = {"soft": P_TR, "multitask": P_TR}   # "distill" est ajouté en partie 9.3


class TweetClassifier(nn.Module):
    def __init__(self, model_name, head="cls", target="soft", n_feats=0, dropout=0.1,
                 cnn_filters=256, cnn_widths=(4, 6, 8)):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.head = head
        self.drop = nn.Dropout(dropout)
        if head == "cnn":
            self.convs = nn.ModuleList([nn.Conv1d(h, cnn_filters, k, padding=k // 2) for k in cnn_widths])
            self.proj = nn.Sequential(nn.Linear(cnn_filters * len(cnn_widths), 256), nn.ReLU())
            d = 256
        else:
            self.proj = nn.Sequential(nn.Linear(h, h), nn.Tanh())
            d = h
        n_out = N_OUT[target]
        self.out = nn.Linear(d, n_out)
        self.aux2 = nn.Linear(d, 4) if target == "multitask" else None
        self.aux3 = nn.Linear(d, 5) if target == "multitask" else None
        self.feat = nn.Linear(n_feats, n_out) if n_feats else None
        if self.feat is not None:
            nn.init.zeros_(self.feat.weight); nn.init.zeros_(self.feat.bias)

    def forward(self, input_ids, attention_mask, feats=None):
        hs = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        if self.head == "cls":
            z = hs[:, 0]
        elif self.head == "mean":
            m = attention_mask.unsqueeze(-1).to(hs.dtype)
            z = (hs * m).sum(1) / m.sum(1).clamp(min=1)
        else:
            x = hs.transpose(1, 2)
            L = x.size(2)
            pooled = []
            for conv in self.convs:
                c = F.relu(conv(x))[:, :, :L]                       # largeur paire : on recoupe à L
                c = c.masked_fill(attention_mask[:, None, :] == 0, -1e4)
                pooled.append(c.max(2).values)
            z = torch.cat(pooled, 1)
        z = self.proj(self.drop(z))
        z = self.drop(z)
        out = {"logits": self.out(z)}
        if self.feat is not None and feats is not None:
            out["logits"] = out["logits"] + self.feat(feats)
        if self.aux2 is not None:
            out["aux2"], out["aux3"] = self.aux2(z), self.aux3(z)
        return out


def p_yes_from_logits(logits, target):
    probs = torch.softmax(logits.float(), dim=-1)
    if N_OUT[target] == 2:
        return probs[:, 1]
    if target == "cls4":
        return 1 - probs[:, 0]
    return (probs * torch.arange(7, device=probs.device) / 6).sum(1)      # cls7


def soft_cross_entropy(logits, target_dist):
    return -(target_dist * F.log_softmax(logits.float(), dim=-1)).sum(-1).mean()


def compute_loss(out, idx, cfg):
    t = cfg["target"]
    logits = out["logits"]
    if t in SOFT_TARGETS:
        p = torch.tensor(SOFT_TARGETS[t][idx], dtype=torch.float, device=device)
        loss = soft_cross_entropy(logits, torch.stack([1 - p, p], 1))
    elif t == "hard":
        p = torch.tensor(Y_TR[idx], dtype=torch.float, device=device)
        loss = soft_cross_entropy(logits, torch.stack([1 - p, p], 1))
    elif t == "cls4":
        loss = soft_cross_entropy(logits, torch.tensor(T2_TR[idx], dtype=torch.float, device=device))
    else:
        loss = F.cross_entropy(logits.float(), torch.tensor(NYES_TR[idx], dtype=torch.long, device=device))
    if t == "multitask":
        loss = loss + cfg["aux_w"] * (
            soft_cross_entropy(out["aux2"], torch.tensor(T2_TR[idx], dtype=torch.float, device=device))
            + F.binary_cross_entropy_with_logits(out["aux3"].float(), torch.tensor(T3_TR[idx], dtype=torch.float, device=device)))
    return loss


AEDA_PUNCT = [".", ";", "?", ":", "!", ","]


def aeda(text, rng):
    words = text.split()
    if not words:
        return text
    n_ins = int(rng.integers(1, max(2, len(words) // 3 + 1)))
    pos = set(rng.choice(len(words), size=min(n_ins, len(words)), replace=False).tolist())
    out = []
    for i, w in enumerate(words):
        if i in pos:
            out.append(AEDA_PUNCT[int(rng.integers(len(AEDA_PUNCT)))])
        out.append(w)
    return " ".join(out)


def param_groups(model, cfg):
    n_layers = model.encoder.config.num_hidden_layers
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        no_wd = p.ndim == 1 or "LayerNorm" in n or "layernorm" in n or n.endswith(".bias")
        if n.startswith("encoder."):
            m = re.search(r"\.layer\.(\d+)\.", n)
            depth = 0 if "embeddings" in n else (int(m.group(1)) + 1 if m else n_layers)
            lr = cfg["lr"] * cfg["llrd"] ** (n_layers - depth)
        elif n.startswith("feat."):
            lr = 1e-3
        else:
            lr = cfg["head_lr"]
        key = (lr, 0.0 if no_wd else 0.01)
        groups.setdefault(key, []).append(p)
    return [{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()]


def encode(tok, texts, max_len):
    enc = tok(list(texts), truncation=True, max_length=max_len, padding=True, return_tensors="pt")
    return enc["input_ids"].to(device), enc["attention_mask"].to(device)


@torch.no_grad()
def predict(model, tok, texts, feats, cfg, bs=128):
    model.eval()
    order = np.argsort([len(t) for t in texts])
    out = np.zeros(len(texts))
    for s in range(0, len(texts), bs):
        idx = order[s:s + bs]
        ids, mask = encode(tok, texts[idx], cfg["max_len"])
        f = torch.tensor(feats[idx], device=device) if cfg["feats"] else None
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            o = model(ids, mask, f)
        out[idx] = p_yes_from_logits(o["logits"], cfg["target"]).cpu().numpy()
    return out


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [15]:
def cfg_key(cfg):
    keys = ["model", "prep", "target", "head", "aeda", "feats", "epochs", "lr", "head_lr", "llrd", "bs", "max_len", "seed", "aux_w"]
    return "_".join(str(cfg[k]).replace("/", "-") for k in keys)


def train_fold(cfg, fold):
    """Entraîne un fold ; met en cache les prédictions (validation et dev) de chaque époque."""
    path = CACHE_DIR / "runs" / cfg_key(cfg) / f"fold{fold}.npz"
    if path.exists():
        return dict(np.load(path))
    path.parent.mkdir(parents=True, exist_ok=True)
    set_seed(cfg["seed"] * 100 + fold)
    rng = np.random.default_rng(cfg["seed"] * 100 + fold)

    tok = get_tokenizer(cfg["model"])
    texts_tr, texts_dev = TEXTS[cfg["prep"]]
    va_idx = np.where(FOLDS == fold)[0]
    tr_idx = np.where(FOLDS != fold)[0]
    if cfg["target"] == "hard":
        tr_idx = tr_idx[~TIE_TR[tr_idx]]

    model = TweetClassifier(cfg["model"], cfg["head"], cfg["target"], n_feats=HFZ_TR.shape[1] if cfg["feats"] else 0).to(device)
    optimizer = torch.optim.AdamW(param_groups(model, cfg), fused=True)
    copies = 2 if cfg["aeda"] else 1
    steps_per_epoch = math.ceil(len(tr_idx) * copies / cfg["bs"])
    total = steps_per_epoch * cfg["epochs"]
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)

    p_va, p_dev, losses = [], [], []
    for epoch in range(cfg["epochs"]):
        model.train()
        items = [(i, False) for i in tr_idx] + ([(i, True) for i in tr_idx] if cfg["aeda"] else [])
        items = [items[j] for j in rng.permutation(len(items))]
        ep_losses = []
        for s in tqdm(range(0, len(items), cfg["bs"]), desc=f"{cfg['name']} fold {fold} époque {epoch + 1}", leave=False):
            batch = items[s:s + cfg["bs"]]
            idx = np.array([i for i, _ in batch])
            texts = [aeda(texts_tr[i], rng) if aug else texts_tr[i] for i, aug in batch]
            ids, mask = encode(tok, texts, cfg["max_len"])
            f = torch.tensor(HFZ_TR[idx], device=device) if cfg["feats"] else None
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                out = model(ids, mask, f)
            loss = compute_loss(out, idx, cfg)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            ep_losses.append(loss.item())
        losses.append(np.mean(ep_losses))
        p_va.append(predict(model, tok, texts_tr[va_idx], HFZ_TR[va_idx], cfg))
        p_dev.append(predict(model, tok, texts_dev, HFZ_DEV, cfg))

    res = dict(va_idx=va_idx, p_va=np.array(p_va), p_dev=np.array(p_dev), loss=np.array(losses))
    np.savez(path, **res)
    del model, optimizer, scheduler
    gc.collect(); torch.cuda.empty_cache()
    return res


EPOCH_TABLES = {}


def collect_folds(fold_fn, n_epochs):
    """Assemble les 5 folds : OOF (chaque tweet prédit par le modèle qui ne l'a pas vu) et dev (moyenne des 5 modèles)."""
    oof = np.full((n_epochs, N_TR), np.nan)
    dev = np.zeros((n_epochs, N_DEV))
    for fold in range(N_FOLDS):
        r = fold_fn(fold)
        oof[:, r["va_idx"]] = r["p_va"]
        dev += r["p_dev"] / N_FOLDS
    return oof, dev


def select_epoch(name, oof, dev, kind, **info):
    """Choisit l'époque sur l'OOF F1 (seuil optimal), à égalité sur la cross-entropy OOF, et enregistre."""
    rows = []
    for e in range(oof.shape[0]):
        thr, f1b = best_threshold(oof[e])
        m = EVAL_TR & ~np.isnan(oof[e])
        rows.append({"époque": e + 1, "OOF F1@0.5": f1_at(Y_TR[m], oof[e][m]), "OOF F1@seuil": f1b,
                     "seuil": thr, "OOF CE": soft_ce(P_TR, oof[e]), "dev F1@0.5": f1_at(Y_DEV[EVAL_DEV], dev[e][EVAL_DEV])})
    table = pd.DataFrame(rows)
    best = int(table.sort_values(["OOF F1@seuil", "OOF CE"], ascending=[False, True]).index[0])
    EPOCH_TABLES[name] = table
    register(name, oof[best], dev[best], kind, epoch=best + 1, **info)
    print(f"{name:22s} époque retenue {best + 1} | OOF F1 {table.loc[best, 'OOF F1@seuil']:.4f} "
          f"| dev F1@0.5 {table.loc[best, 'dev F1@0.5']:.4f}")
    return table


def run_config(cfg):
    t0 = time.time()
    oof, dev = collect_folds(lambda fold: train_fold(cfg, fold), cfg["epochs"])
    select_epoch(cfg["name"], oof, dev, "transformer", cfg=cfg)
    print(f"   ({(time.time() - t0) / 60:.1f} min)")

## 6. Baseline et ablations

### 6.1 La baseline

Même pipeline que le dernier run : Twitter-XLM-R large 2022, preprocessing `light` (celui du run, bugs compris), tête linéaire, batch 16, 128 tokens, entraînée en **hard** et en **soft**, puis **moyenne des deux**. Le dernier run utilisait 1 époque à LR 2e-5 ; le rapport, 3 époques à 2e-5 avec choix de l'époque. On essaie les deux calendriers (et 1e-5 sur 3 époques, plus sûr pour un modèle *large*) et on garde **celui qui a le meilleur OOF** pour la baseline et pour toutes les ablations.

In [16]:
BASE = dict(model="cardiffnlp/twitter-xlm-roberta-large-2022", prep="light", target="soft", head="cls",
            aeda=False, feats=False, epochs=3, lr=1e-5, head_lr=1e-4, llrd=1.0, bs=16, max_len=128, seed=SEED, aux_w=0.5)


def make_cfg(name, **kw):
    c = dict(BASE, **kw)
    c["name"] = name
    return c


def dev_score(name):
    """F1 YES du dev, au seuil choisi sur l'OOF (colonne « dev F1@seuil »)."""
    r = REG[name]
    thr, _ = best_threshold(r["oof"])
    return f1_at(Y_DEV[EVAL_DEV], r["dev"][EVAL_DEV], thr)


# calendriers de la baseline (soft + hard) : NE PAS MODIFIER, les ablations 6.2 en dépendent via le cache
SCHEDULES = {"1ep_lr2e-5": dict(epochs=1, lr=2e-5), "3ep_lr2e-5": dict(epochs=3, lr=2e-5), "3ep_lr1e-5": dict(epochs=3, lr=1e-5)}

# calendriers supplémentaires, soft uniquement : servent à choisir la configuration de la partie 7
SOFT_SCHEDULES = {
    "3ep_lr1.5e-5": dict(epochs=3, lr=1.5e-5),
    "3ep_lr3e-5": dict(epochs=3, lr=3e-5),
    "4ep_lr1.5e-5": dict(epochs=4, lr=1.5e-5),
    "4ep_lr2e-5": dict(epochs=4, lr=2e-5),
    "3ep_lr3e-5_llrd0.9": dict(epochs=3, lr=3e-5, llrd=0.9),
    "4ep_lr3e-5_llrd0.9": dict(epochs=4, lr=3e-5, llrd=0.9),
    "3ep_lr3e-5_bs32": dict(epochs=3, lr=3e-5, bs=32),
}
RUN_SOFT_SCHEDULES = True
SCHED_REF = "sched_3ep_lr2e-5_soft"            # meilleur soft de la première série
BEST_SOFT_KW = dict(epochs=3, lr=2e-5)         # valeur par défaut si rien ne tourne

if RUN_TRANSFORMERS:
    # --- baseline : inchangé ---
    for sched, kw in SCHEDULES.items():
        for target in ["soft", "hard"]:
            run_config(make_cfg(f"sched_{sched}_{target}", target=target, **kw))
    for sched in SCHEDULES:
        a, b = f"sched_{sched}_soft", f"sched_{sched}_hard"
        register(f"sched_{sched}_moyenne", (REG[a]["oof"] + REG[b]["oof"]) / 2, (REG[a]["dev"] + REG[b]["dev"]) / 2, "transformer")
    best_sched = max(SCHEDULES, key=lambda s: oof_score(f"sched_{s}_moyenne"))
    BASE.update(SCHEDULES[best_sched])
    print("Calendrier retenu pour la baseline et les ablations :", best_sched, SCHEDULES[best_sched])
    for t in ["soft", "hard"]:
        REG[f"base_{t}"] = REG[f"sched_{best_sched}_{t}"]
    REG["baseline"] = REG[f"sched_{best_sched}_moyenne"]
    display(summary([n for n in REG if n.startswith("sched_")] + ["baseline"]))

    # --- calendriers soft supplémentaires, choix sur le dev ---
    if RUN_SOFT_SCHEDULES:
        for sched, kw in SOFT_SCHEDULES.items():
            run_config(make_cfg(f"sched_{sched}_soft", target="soft", **kw))
    ALL_SOFT = {**SCHEDULES, **SOFT_SCHEDULES}
    soft_names = [f"sched_{s}_soft" for s in ALL_SOFT if f"sched_{s}_soft" in REG]
    display(summary(soft_names, sort="dev F1@seuil", ref=SCHED_REF))

    best_soft = max(soft_names, key=dev_score)
    if dev_score(best_soft) < dev_score(SCHED_REF) + 0.003:   # on ne change que pour un gain d'au moins 0,3 point
        best_soft = SCHED_REF
    BEST_SOFT_KW = ALL_SOFT[best_soft.removeprefix("sched_").removesuffix("_soft")]
    print(f"Calendrier soft retenu pour la partie 7 : {best_soft} | dev F1 {dev_score(best_soft):.4f} | {BEST_SOFT_KW}")

sched_1ep_lr2e-5_soft  époque retenue 1 | OOF F1 0.8442 | dev F1@0.5 0.8685
   (0.0 min)
sched_1ep_lr2e-5_hard  époque retenue 1 | OOF F1 0.8416 | dev F1@0.5 0.8623
   (0.0 min)
sched_3ep_lr2e-5_soft  époque retenue 3 | OOF F1 0.8512 | dev F1@0.5 0.8833
   (0.0 min)
sched_3ep_lr2e-5_hard  époque retenue 3 | OOF F1 0.8469 | dev F1@0.5 0.8578
   (0.0 min)
sched_3ep_lr1e-5_soft  époque retenue 3 | OOF F1 0.8506 | dev F1@0.5 0.8776
   (0.0 min)
sched_3ep_lr1e-5_hard  époque retenue 3 | OOF F1 0.8455 | dev F1@0.5 0.8635
   (0.0 min)
Calendrier retenu pour la baseline et les ablations : 3ep_lr1e-5 {'epochs': 3, 'lr': 1e-05}


,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
sched_3ep_lr2e-5_soft,transformer,0.8487,0.47,0.8512,0.5377,0.8833,0.8829,0.8602,0.9030,0.8456,0.9108,0.5008,0.0150,0.0180
sched_3ep_lr1e-5_soft,transformer,0.8506,0.50,0.8506,0.5369,0.8776,0.8776,0.8543,0.8991,0.8454,0.9017,0.5027,0.0098,0.1075
sched_3ep_lr1e-5_moyenne,transformer,0.8450,0.27,0.8505,0.6054,0.8705,0.8678,0.8444,0.8896,0.8592,0.8743,0.5541,0.0000,1.0000
baseline,transformer,0.8450,0.27,0.8505,0.6054,0.8705,0.8678,0.8444,0.8896,0.8592,0.8743,0.5541,NaN,NaN
sched_3ep_lr2e-5_moyenne,transformer,0.8492,0.46,0.8502,0.6060,0.8626,0.8678,0.8435,0.8906,0.8299,0.8962,0.5533,-0.0000,0.5095
sched_3ep_lr2e-5_hard,transformer,0.8456,0.46,0.8469,1.3701,0.8578,0.8597,0.8343,0.8831,0.8281,0.8833,1.0617,-0.0082,0.8205
sched_1ep_lr2e-5_moyenne,transformer,0.8446,0.47,0.8467,0.5511,0.8658,0.8669,0.8422,0.8894,0.8440,0.8842,0.5195,-0.0010,0.5485
sched_3ep_lr1e-5_hard,transformer,0.8413,0.21,0.8455,1.2713,0.8635,0.8671,0.8429,0.8893,0.8564,0.8750,1.0034,-0.0008,0.5545
sched_1ep_lr2e-5_soft,transformer,0.8415,0.48,0.8442,0.5270,0.8685,0.8706,0.8464,0.8929,0.8469,0.8885,0.5020,0.0028,0.3765


sched_3ep_lr1.5e-5_soft époque retenue 3 | OOF F1 0.8538 | dev F1@0.5 0.8742
   (0.0 min)
sched_3ep_lr3e-5_soft  époque retenue 3 | OOF F1 0.8509 | dev F1@0.5 0.8779
   (0.0 min)
sched_4ep_lr1.5e-5_soft époque retenue 2 | OOF F1 0.8491 | dev F1@0.5 0.8616
   (0.0 min)
sched_4ep_lr2e-5_soft  époque retenue 4 | OOF F1 0.8507 | dev F1@0.5 0.8700
   (0.0 min)
sched_3ep_lr3e-5_llrd0.9_soft époque retenue 3 | OOF F1 0.8521 | dev F1@0.5 0.8713
   (0.0 min)
sched_4ep_lr3e-5_llrd0.9_soft époque retenue 3 | OOF F1 0.8505 | dev F1@0.5 0.8783
   (0.0 min)
sched_3ep_lr3e-5_bs32_soft époque retenue 3 | OOF F1 0.7865 | dev F1@0.5 0.8790
   (0.0 min)


,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
sched_3ep_lr2e-5_soft,transformer,0.8487,0.47,0.8512,0.5377,0.8833,0.8829,0.8602,0.9030,0.8456,0.9108,0.5008,NaN,NaN
sched_3ep_lr3e-5_soft,transformer,0.8476,0.47,0.8509,0.5420,0.8779,0.8824,0.8599,0.9036,0.8529,0.9049,0.5057,-0.0004,0.5185
sched_4ep_lr3e-5_llrd0.9_soft,transformer,0.8443,0.45,0.8505,0.5360,0.8783,0.8818,0.8594,0.9026,0.8499,0.9060,0.5038,-0.0010,0.5575
sched_4ep_lr2e-5_soft,transformer,0.8425,0.43,0.8507,0.5439,0.8700,0.8787,0.8558,0.8996,0.8494,0.9009,0.5054,-0.0041,0.7415
sched_3ep_lr1e-5_soft,transformer,0.8506,0.50,0.8506,0.5369,0.8776,0.8776,0.8543,0.8991,0.8454,0.9017,0.5027,-0.0052,0.8225
sched_4ep_lr1.5e-5_soft,transformer,0.8420,0.43,0.8491,0.5297,0.8616,0.8761,0.8535,0.8979,0.8515,0.8947,0.5004,-0.0068,0.9120
sched_3ep_lr3e-5_bs32_soft,transformer,0.7589,0.46,0.7865,0.5641,0.8790,0.8743,0.8504,0.8962,0.8421,0.8985,0.5004,-0.0085,0.9245
sched_3ep_lr3e-5_llrd0.9_soft,transformer,0.8516,0.48,0.8521,0.5390,0.8713,0.8743,0.8508,0.8963,0.8418,0.8987,0.5072,-0.0085,0.9245
sched_3ep_lr1.5e-5_soft,transformer,0.8538,0.50,0.8538,0.5370,0.8742,0.8742,0.8501,0.8970,0.8424,0.8979,0.5025,-0.0087,0.9575


Calendrier soft retenu pour la partie 7 : sched_3ep_lr2e-5_soft | dev F1 0.8829 | {'epochs': 3, 'lr': 2e-05}


### 6.2 Ablations : un seul facteur change à la fois

Chaque variante modifie **un** réglage de `base_soft` (cible soft, calendrier retenu) ; on la compare donc à `base_soft`, et pas à la moyenne hard + soft.

| Nom | Ce qui change | Question |
|---|---|---|
| `prep_v2` | preprocessing corrigé | les bugs corrigés et le découpage des hashtags aident-ils ? |
| `aeda` | AEDA | l'augmentation aide-t-elle un modèle *large* ? |
| `cls4` | 4 classes (NO / DIRECT / REPORTED / JUDGEMENTAL) | **multi-classes vs binaire** : voir l'intention aide-t-il à reconnaître les tweets qui rapportent ou dénoncent ? |
| `cls7` | 7 classes (nombre de votes YES) | **multi-classes vs binaire** : modéliser le désaccord aide-t-il ? |
| `multitask` | soft + têtes tâche 2 et tâche 3 | les annotations des tâches 2 et 3 apportent-elles un signal ? |
| `head_cnn` | tête MultiCNN | **CNN vs linéaire** |

In [17]:
EXPERIMENTS = [
    make_cfg("prep_v2", prep="v2"),
    make_cfg("aeda", aeda=True),
    make_cfg("cls4", target="cls4"),
    make_cfg("cls7", target="cls7"),
    make_cfg("multitask", target="multitask"),
    make_cfg("head_cnn", head="cnn", head_lr=1e-3),
]
RUN_EXPERIMENT = {c["name"]: True for c in EXPERIMENTS}   # mettre False pour sauter une configuration

if RUN_TRANSFORMERS:
    for c in EXPERIMENTS:
        if RUN_EXPERIMENT[c["name"]]:
            run_config(c)
    display(summary(["base_soft"] + [c["name"] for c in EXPERIMENTS if c["name"] in REG], ref="base_soft"))

prep_v2                époque retenue 3 | OOF F1 0.8550 | dev F1@0.5 0.8719
   (0.0 min)
aeda                   époque retenue 3 | OOF F1 0.8502 | dev F1@0.5 0.8825
   (0.0 min)
cls4                   époque retenue 3 | OOF F1 0.8511 | dev F1@0.5 0.8739
   (0.0 min)
cls7                   époque retenue 3 | OOF F1 0.8498 | dev F1@0.5 0.8660
   (0.0 min)
multitask              époque retenue 3 | OOF F1 0.8520 | dev F1@0.5 0.8736
   (0.0 min)
head_cnn               époque retenue 2 | OOF F1 0.8489 | dev F1@0.5 0.8744
   (0.0 min)


,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
prep_v2,transformer,0.8525,0.46,0.8550,0.5438,0.8719,0.8734,0.8502,0.8953,0.8458,0.8943,0.5128,-0.0042,0.7875
multitask,transformer,0.8511,0.51,0.8520,0.5349,0.8736,0.8733,0.8491,0.8953,0.8355,0.9014,0.5060,-0.0043,0.7890
cls4,transformer,0.8462,0.45,0.8511,0.5358,0.8739,0.8692,0.8466,0.8914,0.8304,0.8981,0.5077,-0.0084,0.9215
base_soft,transformer,0.8506,0.50,0.8506,0.5369,0.8776,0.8776,0.8543,0.8991,0.8454,0.9017,0.5027,NaN,NaN
aeda,transformer,0.8491,0.45,0.8502,0.5486,0.8825,0.8762,0.8540,0.8987,0.8593,0.8889,0.5066,-0.0014,0.5780
cls7,transformer,0.8488,0.48,0.8498,0.5279,0.8660,0.8671,0.8421,0.8894,0.8418,0.8859,0.4990,-0.0105,0.9905
head_cnn,transformer,0.8435,0.47,0.8489,0.5229,0.8744,0.8719,0.8472,0.8932,0.8440,0.8927,0.4991,-0.0058,0.8585


Courbes par époque (OOF et dev) : sur-apprentissage éventuel, et l'époque choisie sur l'OOF est-elle aussi la bonne sur le dev ?

In [18]:
pd.concat({n: t.set_index("époque") for n, t in EPOCH_TABLES.items()}).round(4)

OOF F1@0.5  OOF F1@seuil  seuil  OOF CE  dev F1@0.5
                              époque                                                     
sched_1ep_lr2e-5_soft         1           0.8415        0.8442   0.48  0.5270      0.8685
sched_1ep_lr2e-5_hard         1           0.8380        0.8416   0.34  0.7215      0.8623
sched_3ep_lr2e-5_soft         1           0.8370        0.8387   0.48  0.5341      0.8699
                              2           0.8474        0.8478   0.49  0.5330      0.8823
                              3           0.8487        0.8512   0.47  0.5377      0.8833
sched_3ep_lr2e-5_hard         1           0.8244        0.8329   0.42  0.6896      0.8644
                              2           0.8335        0.8391   0.22  0.8405      0.8578
                              3           0.8456        0.8469   0.46  1.3701      0.8578
sched_3ep_lr1e-5_soft         1           0.8332        0.8386   0.42  0.5348      0.8763
                              2           0.8452        0.8484   0.46  0.5318      0.8739
                              3           0.8506        0.8506   0.50  0.5369      0.8776
sched_3ep_lr1e-5_hard         1           0.8289        0.8289   0.50  0.7461      0.8632
                              2           0.8344        0.8409   0.34  0.8501      0.8683
                              3           0.8413        0.8455   0.21  1.2713      0.8635
sched_3ep_lr1.5e-5_soft       1           0.8405        0.8405   0.50  0.5348      0.8746
                              2           0.8495        0.8510   0.47  0.5283      0.8692
                              3           0.8538        0.8538   0.50  0.5370      0.8742
sched_3ep_lr3e-5_soft         1           0.8283        0.8300   0.48  0.5396      0.8678
                              2           0.8471        0.8479   0.48  0.5355      0.8811
                              3           0.8476        0.8509   0.47  0.5420      0.8779
sched_4ep_lr1.5e-5_soft       1           0.8306        0.8318   0.48  0.5395      0.8738
                              2           0.8420        0.8491   0.43  0.5297      0.8616
                              3           0.8418        0.8462   0.43  0.5355      0.8744
                              4           0.8483        0.8483   0.50  0.5407      0.8758
sched_4ep_lr2e-5_soft         1           0.8267        0.8316   0.44  0.5436      0.8613
                              2           0.8441        0.8487   0.45  0.5282      0.8701
                              3           0.8432        0.8466   0.45  0.5386      0.8744
                              4           0.8425        0.8507   0.43  0.5439      0.8700
sched_3ep_lr3e-5_llrd0.9_soft 1           0.8399        0.8409   0.48  0.5311      0.8661
                              2           0.8468        0.8499   0.45  0.5290      0.8735
                              3           0.8516        0.8521   0.48  0.5390      0.8713
sched_4ep_lr3e-5_llrd0.9_soft 1           0.8334        0.8349   0.47  0.5358      0.8709
                              2           0.8458        0.8491   0.46  0.5276      0.8724
                              3           0.8443        0.8505   0.45  0.5360      0.8783
                              4           0.8478        0.8487   0.46  0.5403      0.8701
sched_3ep_lr3e-5_bs32_soft    1           0.7468        0.7632   0.38  0.5662      0.8717
                              2           0.7563        0.7861   0.44  0.5570      0.8664
                              3           0.7589        0.7865   0.46  0.5641      0.8790
prep_v2                       1           0.8429        0.8444   0.49  0.5329      0.8685
                              2           0.8483        0.8503   0.47  0.5334      0.8764
                              3           0.8525        0.8550   0.46  0.5438      0.8719
aeda                          1           0.8422        0.8433   0.51  0.5390      0.8709
                              2           0.8441        0.8473   0.43  0.5465 

**Multi-classes vs binaire, en détail.** Exactitude par intention majoritaire et par nombre de votes YES : le gain (ou la perte) des sorties multi-classes vient-il des tweets REPORTED / JUDGEMENTAL ?

In [19]:
intent_tr = np.array([TASK2[1 + int(np.argmax(d[1:]))] if d[0] < 0.5 else "NO" for d in T2_TR])
rows = []
for name in [n for n in ["base_soft", "base_hard", "cls4", "cls7", "multitask"] if n in REG]:
    thr, _ = best_threshold(REG[name]["oof"])
    pred = REG[name]["oof"] >= thr
    row = {"modèle": name}
    for it in TASK2:
        m = EVAL_TR & (intent_tr == it)
        row[f"exactitude {it} (n={m.sum()})"] = (pred[m] == Y_TR[m]).mean()
    for v in range(7):
        m = EVAL_TR & (NYES_TR == v)
        if m.any():
            row[f"{v} votes"] = (pred[m] == Y_TR[m]).mean()
    rows.append(row)
pd.DataFrame(rows).set_index("modèle").round(3)

,exactitude NO (n=3367),exactitude DIRECT (n=1625),exactitude REPORTED (n=646),exactitude JUDGEMENTAL (n=426),0 votes,1 votes,2 votes,4 votes,5 votes,6 votes
modèle,,,,,,,,,,
base_soft,0.851,0.876,0.881,0.876,0.967,0.852,0.660,0.777,0.894,0.969
base_hard,0.826,0.890,0.901,0.883,0.957,0.829,0.602,0.795,0.909,0.979
cls4,0.830,0.898,0.909,0.885,0.957,0.839,0.604,0.803,0.922,0.977
cls7,0.851,0.874,0.887,0.866,0.967,0.862,0.647,0.771,0.893,0.972
multitask,0.857,0.874,0.882,0.866,0.967,0.862,0.667,0.767,0.900,0.966


## 7. Meilleure combinaison et plusieurs seeds

Pour chaque facteur, on garde la variante dont l'OOF F1 (au seuil optimal) dépasse `base_soft` d'au moins 0,3 point ; sinon la valeur de la baseline. On obtient la configuration `combo`, entraînée avec **trois seeds**, plus sa version hard. La nouvelle pipeline `combo_pipeline` reproduit la recette de la baseline (moyenne hard + soft), avec les meilleurs réglages et trois seeds côté soft.

In [20]:
combo_kw = dict(BEST_SOFT_KW)   # meilleur calendrier soft (partie 6.3) ; prep light, pas d'AEDA, tête cls
SEEDS = [SEED, SEED + 1, SEED + 2, SEED + 3, SEED + 4]


def with_target(kw, target):   # gardée pour la partie 9.3
    kw = {**kw, "target": target}
    if kw.get("head") != "cnn":
        kw.pop("head_lr", None)
    return kw


if RUN_COMBO:
    oofs, devs = [], []
    for s in SEEDS:
        cfg = make_cfg(f"soft_s{s}", **combo_kw, target="soft", seed=s)
        oof, dev = collect_folds(lambda fold: train_fold(cfg, fold), cfg["epochs"])
        select_epoch(cfg["name"], oof, dev, "transformer", cfg=cfg)
        oofs.append(oof); devs.append(dev)

    # moyenne des seeds, puis choix de l'époque sur l'OOF de la moyenne
    select_epoch("combo_seeds", np.mean(oofs, 0), np.mean(devs, 0), "transformer", seeds=SEEDS)
    display(EPOCH_TABLES["combo_seeds"].round(4))
    display(summary(["baseline", "base_soft"] + [f"soft_s{s}" for s in SEEDS] + ["combo_seeds"]))

soft_s42               époque retenue 3 | OOF F1 0.8512 | dev F1@0.5 0.8833
soft_s43               époque retenue 3 | OOF F1 0.8533 | dev F1@0.5 0.8699
soft_s44               époque retenue 3 | OOF F1 0.8467 | dev F1@0.5 0.8758
soft_s45               époque retenue 3 | OOF F1 0.8522 | dev F1@0.5 0.8735
soft_s46               époque retenue 3 | OOF F1 0.8457 | dev F1@0.5 0.8739
combo_seeds            époque retenue 3 | OOF F1 0.8564 | dev F1@0.5 0.8786


,époque,OOF F1@0.5,OOF F1@seuil,seuil,OOF CE,dev F1@0.5
0,1,0.8504,0.8524,0.48,0.5221,0.8731
1,2,0.8527,0.8540,0.46,0.5187,0.8747
2,3,0.8534,0.8564,0.46,0.5284,0.8786


,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
combo_seeds,transformer,0.8534,0.46,0.8564,0.5284,0.8786,0.8778,0.8555,0.8995,0.8471,0.9011,0.5027,0.0100,0.0645
soft_s43,transformer,0.8484,0.44,0.8533,0.5390,0.8699,0.8770,0.8537,0.8983,0.8536,0.8947,0.5066,0.0092,0.0605
soft_s45,transformer,0.8518,0.46,0.8522,0.5366,0.8735,0.8836,0.8614,0.9046,0.8550,0.9053,0.5029,0.0158,0.0100
soft_s42,transformer,0.8487,0.47,0.8512,0.5377,0.8833,0.8829,0.8602,0.9030,0.8456,0.9108,0.5008,0.0150,0.0180
base_soft,transformer,0.8506,0.50,0.8506,0.5369,0.8776,0.8776,0.8543,0.8991,0.8454,0.9017,0.5027,0.0098,0.1075
baseline,transformer,0.8450,0.27,0.8505,0.6054,0.8705,0.8678,0.8444,0.8896,0.8592,0.8743,0.5541,NaN,NaN
soft_s44,transformer,0.8437,0.47,0.8467,0.5407,0.8758,0.8761,0.8532,0.8981,0.8413,0.9025,0.5062,0.0082,0.1295
soft_s46,transformer,0.8447,0.48,0.8457,0.5439,0.8739,0.8747,0.8509,0.8970,0.8462,0.8962,0.5076,0.0069,0.1870


### 8. LLM

Quatre façons d'utiliser un LLM de 7B, de la moins chère à la plus chère :

1. **Prompting few-shot** (9.1) : le LLM lit la définition EXIST, 8 exemples, puis le tweet. On ne le fait **pas générer** : on lit, au premier token de sa réponse, la probabilité relative de « YES » et de « NO », ce qui donne une probabilité continue P(YES) qu'on peut seuiller, calibrer et empiler avec BERT.
2. **Justifications générées** (9.2, CIMAT-GTO, 2e en 2025) : le LLM écrit, sans connaître l'étiquette, un argument pour et un argument contre le caractère sexiste du tweet ; on ajoute ce texte au tweet avant de fine-tuner XLM-R.
3. **Fine-tuning LoRA du LLM** (9.3, Mario, 1er en 2025) : le LLM 7B devient un classifieur ; on entraîne des adaptateurs LoRA sur la probabilité de « YES » au premier token, avec la cible soft. Même protocole en 5 folds que XLM-R.
4. **Distillation** (9.3) : XLM-R est ré-entraîné sur une cible qui mélange la part de YES et la prédiction OOF du LLM fine-tuné.

Choix communs :

- **Définition** : celle d'EXIST, en insistant sur le point qui pose problème au fine-tuning : un tweet qui *rapporte* ou *dénonce* du sexisme compte comme YES.
- **Few-shot** : 8 exemples du train (4 YES, 4 NO), **choisis par similarité** avec le tweet (embeddings e5), les plus proches en dernier. Pour un tweet du train, les exemples viennent des autres folds, ce qui rend la prédiction utilisable en OOF.
- **Modèle** : `Qwen2.5-7B-Instruct` pour les quatre usages (bon en espagnol, sans autorisation d'accès à demander).

### 8.1 Prompting few-shot

In [21]:
DEFINITION = """You are an expert annotator for the EXIST 2024 shared task on sexism identification in tweets written in English or Spanish.

Question asked to the annotators: is the tweet sexist itself, does it describe a situation in which discrimination against women occurs, or does it criticise a sexist behaviour?
Answer YES in all three cases:
- DIRECT: the tweet is sexist itself (it insults, belittles, objectifies, stereotypes or threatens women because they are women, defends gender roles, attacks feminism or denies women's rights).
- REPORTED: the tweet reports a sexist situation suffered by a woman or women (personal experience of harassment or abuse, news about gender violence, discrimination, etc.).
- JUDGEMENTAL: the tweet describes or condemns sexist situations or behaviours (criticism of machismo, of the wage gap, of mansplaining, of harassment, etc.).
Answer NO if the tweet is not about sexism: for instance a word of the topic used in another sense, an insult or vulgarity that does not target women as women, a neutral or positive mention of women, advertising, or unrelated content."""


def build_prompt(query, demos):
    parts = [DEFINITION]
    if demos:
        parts.append("Examples labelled by the majority vote of six human annotators:")
        parts += [f"Tweet: {t[:500]}\nAnswer: {lab}" for t, lab in demos]
    parts.append(f"Classify the following tweet.\nTweet: {query[:700]}\nAnswer with a single word: YES or NO.")
    return "\n\n".join(parts)


LLM_TEXTS_TR, LLM_TEXTS_DEV = TEXTS["xlmt"]


def pick_demos(sims, allowed, k):
    """k/2 exemples YES et k/2 NO, les plus similaires, ordonnés du moins au plus proche."""
    demos = []
    for lab_value, lab in [(1, "YES"), (0, "NO")]:
        s = np.where(allowed & EVAL_TR & (Y_TR == lab_value), sims, -np.inf)
        top = np.argpartition(-s, k // 2)[:k // 2]
        demos += [(sims[j], LLM_TEXTS_TR[j], lab) for j in top]
    return [(t, lab) for _, t, lab in sorted(demos)]


def make_prompts(k):
    tr = [build_prompt(LLM_TEXTS_TR[i], pick_demos(SIM_TR[i], allowed_neighbours(i), k)) for i in range(N_TR)]
    dv = [build_prompt(LLM_TEXTS_DEV[i], pick_demos(SIM_DEV[i], allowed_neighbours(), k)) for i in range(N_DEV)]
    return tr, dv


print(build_prompt(LLM_TEXTS_DEV[0], pick_demos(SIM_DEV[0], allowed_neighbours(), 4)))

You are an expert annotator for the EXIST 2024 shared task on sexism identification in tweets written in English or Spanish.

Question asked to the annotators: is the tweet sexist itself, does it describe a situation in which discrimination against women occurs, or does it criticise a sexist behaviour?
Answer YES in all three cases:
- DIRECT: the tweet is sexist itself (it insults, belittles, objectifies, stereotypes or threatens women because they are women, defends gender roles, attacks feminism or denies women's rights).
- REPORTED: the tweet reports a sexist situation suffered by a woman or women (personal experience of harassment or abuse, news about gender violence, discrimination, etc.).
- JUDGEMENTAL: the tweet describes or condemns sexist situations or behaviours (criticism of machismo, of the wage gap, of mansplaining, of harassment, etc.).
Answer NO if the tweet is not about sexism: for instance a word of the topic used in another sense, an insult or vulgarity that does not 

In [22]:
import gc
import torch

# Suppression des variables volumineuses potentiellement restées en mémoire
for _name in ["llm_model", "r_model", "ft_base", "llm_tok", "r_tok", "ft_tok"]:
    globals().pop(_name, None)

gc.collect()
torch.cuda.empty_cache()


def vram(msg=""):
    print(f"{msg} VRAM allouée : {torch.cuda.memory_allocated(0) / 1e9:.2f} Go | "
          f"réservée : {torch.cuda.memory_reserved(0) / 1e9:.2f} Go")


vram("Après nettoyage —")
if torch.cuda.memory_allocated(0) > 2e9:
    print("⚠ Plus de 2 Go encore alloués : d'autres modèles sont en mémoire. "
          "Si l'OOM persiste, redémarre le noyau (les résultats déjà calculés sont en cache).")

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"   # ou "Qwen/Qwen3-8B"
LLM_TOKEN_BUDGET = 4_000        # scoring (8 000 pour le 3B)
LLM_GEN_TOKEN_BUDGET = 6_000    # justifications (12 000 pour le 3B)
LLM_FT_4BIT = True              # QLoRA : indispensable pour fine-tuner un 7B sur 22 Go
print(f"Le modèle LLM a été configuré sur : {LLM_NAME}")

Après nettoyage — VRAM allouée : 0.00 Go | réservée : 0.00 Go
Le modèle LLM a été configuré sur : Qwen/Qwen2.5-7B-Instruct


In [23]:
def load_llm(name, four_bit=False):
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = dict(torch_dtype=torch.bfloat16, device_map="cuda", attn_implementation="sdpa")
    if four_bit:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.bfloat16,
                                                       bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(name, **kw).eval()
    vram(f"{name}{' (4 bits)' if four_bit else ''} chargé —")
    return tok, model


def chat_text(tok, prompt):
    msgs = [{"role": "user", "content": prompt}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def answer_token_ids(tok, words):
    return sorted({tok.encode(w, add_special_tokens=False)[0] for w in words})


def left_pad(tok, seqs):
    L = max(len(e) for e in seqs)
    ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for r, e in enumerate(seqs):
        ids[r, L - len(e):] = torch.tensor(e); mask[r, L - len(e):] = 1
    pos = (mask.cumsum(1) - 1).clamp(min=0)        # positions correctes malgré le padding à gauche
    return ids.to(device), mask.to(device), pos.to(device)


def last_token_logits(causal_lm, ids, mask, pos):
    """Logits du dernier token seulement ; use_cache=False évite de construire un cache KV inutile."""
    h = causal_lm.get_decoder()(input_ids=ids, attention_mask=mask, position_ids=pos,
                                use_cache=False).last_hidden_state[:, -1]
    return causal_lm.lm_head(h).float()


def adaptive_batches(enc, order, token_budget, step, desc=None, extra_len=0):
    """Traite `order` (trié du plus long au plus court) par lots d'environ `token_budget` tokens.
    En cas d'OOM, le lot est divisé par deux et relancé ; le budget réduit est conservé ensuite."""
    pbar = tqdm(total=len(order), desc=desc, disable=desc is None)
    i = 0
    while i < len(order):
        L = len(enc[order[i]]) + extra_len
        n = max(1, token_budget // L)
        idx = order[i:i + n]
        oom = False
        try:
            step(idx)
        except torch.cuda.OutOfMemoryError:
            if n == 1:
                raise
            oom = True
        if oom:                                    # nettoyage hors du bloc except (libère le traceback)
            gc.collect(); torch.cuda.empty_cache()
            token_budget = max(L, (n // 2) * L)
            print(f"  OOM : lot de {n} → {max(1, token_budget // L)} (budget {token_budget} tokens)")
            continue
        i += len(idx); pbar.update(len(idx))
    pbar.close()


@torch.no_grad()
def llm_p_yes(tok, model, prompts, token_budget=LLM_TOKEN_BUDGET):
    yes_ids = answer_token_ids(tok, ["YES", "Yes", "yes", " YES", " Yes"])
    no_ids = answer_token_ids(tok, ["NO", "No", "no", " NO", " No"])
    assert not set(yes_ids) & set(no_ids), (yes_ids, no_ids)
    enc = [tok(chat_text(tok, p), add_special_tokens=False)["input_ids"] for p in prompts]
    order = np.argsort([-len(e) for e in enc])          # les plus longs d'abord : un éventuel OOM arrive tout de suite
    out = np.zeros(len(prompts))

    def step(idx):
        logits = last_token_logits(model, *left_pad(tok, [enc[j] for j in idx]))
        diff = torch.logsumexp(logits[:, yes_ids], 1) - torch.logsumexp(logits[:, no_ids], 1)
        out[idx] = torch.sigmoid(diff).cpu().numpy()

    adaptive_batches(enc, order, token_budget, step, desc="LLM")
    return out


if RUN_LLM_PROMPT:
    path = CACHE_DIR / f"llm_few_shot_k{LLM_FEWSHOT_K}_{LLM_NAME.replace('/', '-')}.npz"
    if not path.exists():
        llm_tok, llm_model = load_llm(LLM_NAME)
        prompts_tr, prompts_dev = make_prompts(LLM_FEWSHOT_K)
        p_tr = llm_p_yes(llm_tok, llm_model, prompts_tr)
        p_dev = llm_p_yes(llm_tok, llm_model, prompts_dev)
        np.savez(path, tr=p_tr, dev=p_dev)
        del llm_model; gc.collect(); torch.cuda.empty_cache()
    z = np.load(path)
    register("llm_few_shot", z["tr"], z["dev"], "LLM", model=LLM_NAME)
    display(summary([n for n in ["llm_few_shot", "base_soft", REFERENCE] if n in REG]))

PROMPT_LLMS = [n for n in REG if REG[n]["kind"] == "LLM"]
if PROMPT_LLMS and REFERENCE in REG:
    best_llm = max(PROMPT_LLMS, key=lambda n: best_threshold(REG[n]["oof"])[1])
    tb, _ = best_threshold(REG[REFERENCE]["oof"]); tl, _ = best_threshold(REG[best_llm]["oof"])
    b_ok = (REG[REFERENCE]["oof"] >= tb) == Y_TR
    l_ok = (REG[best_llm]["oof"] >= tl) == Y_TR
    unsure = np.abs(REG[REFERENCE]["oof"] - tb) < 0.15
    m = EVAL_TR
    print("Meilleur LLM en prompting :", best_llm)
    print(pd.crosstab(pd.Series(b_ok[m], name="BERT correct"), pd.Series(l_ok[m], name="LLM correct")))
    print(f"Sur les tweets où BERT hésite (|p - seuil| < 0,15, n = {(m & unsure).sum()}) : "
          f"BERT {b_ok[m & unsure].mean():.3f}, LLM {l_ok[m & unsure].mean():.3f}")

,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
base_soft,transformer,0.8506,0.50,0.8506,0.5369,0.8776,0.8776,0.8543,0.8991,0.8454,0.9017,0.5027,0.0098,0.1075
baseline,transformer,0.8450,0.27,0.8505,0.6054,0.8705,0.8678,0.8444,0.8896,0.8592,0.8743,0.5541,NaN,NaN
llm_few_shot,LLM,0.7642,0.80,0.7660,2.2108,0.7836,0.7984,0.7709,0.8241,0.7838,0.8098,2.1069,-0.0694,1.0000


Meilleur LLM en prompting : llm_few_shot
LLM correct   False  True 
BERT correct              
False           520    351
True            943   4250
Sur les tweets où BERT hésite (|p - seuil| < 0,15, n = 885) : BERT 0.708, LLM 0.554


Le LLM se trompe-t-il sur les mêmes tweets que BERT ? S'il est correct là où BERT hésite, le routage et le stacking de la partie 10 en tireront parti.

### 8.2 Justifications générées par le LLM, puis XLM-R

Pour chaque tweet (train et dev), le LLM écrit **sans connaître l'étiquette** un argument pour et un argument contre le caractère sexiste du tweet, au sens d'EXIST. Le texte est ajouté après le tweet (`tweet | Analysis: …`) et XLM-R est fine-tuné sur cette entrée enrichie (256 tokens), avec les réglages de `combo`. Il n'y a pas de fuite : le LLM n'a jamais vu les étiquettes.

In [24]:
RATIONALE_INSTRUCTION = DEFINITION + """

Tweet: {tweet}

Do not give a final verdict. Write exactly two short sentences: first the strongest argument that this tweet is sexist under the definition above (remember that reporting or criticising sexism counts), then the strongest argument that it is not. Mention irony or sarcasm if present."""


@torch.no_grad()
def generate_rationales(tok, model, tweets, token_budget=LLM_GEN_TOKEN_BUDGET, max_new_tokens=110):
    prompts = [chat_text(tok, RATIONALE_INSTRUCTION.format(tweet=t[:700])) for t in tweets]
    enc = [tok(p, add_special_tokens=False)["input_ids"] for p in prompts]
    order = np.argsort([-len(e) for e in enc])
    out = [""] * len(tweets)

    def step(idx):
        ids, mask, _ = left_pad(tok, [enc[j] for j in idx])
        gen = model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
        for j, t in zip(idx, texts):
            out[j] = " ".join(t.split())

    adaptive_batches(enc, order, token_budget, step, desc="justifications", extra_len=max_new_tokens)
    return out


rat_cache = CACHE_DIR / f"rationales_{LLM_NAME.replace('/', '-')}.json"
if RUN_LLM_RATIONALES:
    if rat_cache.exists():
        RATIONALES = json.load(open(rat_cache, encoding="utf-8"))
    else:
        r_tok, r_model = load_llm(LLM_NAME)
        RATIONALES = {"tr": generate_rationales(r_tok, r_model, list(LLM_TEXTS_TR)),
                      "dev": generate_rationales(r_tok, r_model, list(LLM_TEXTS_DEV))}
        json.dump(RATIONALES, open(rat_cache, "w", encoding="utf-8"), ensure_ascii=False)
        del r_model; gc.collect(); torch.cuda.empty_cache()
    TEXTS["rationale"] = (np.array([f"{t} | Analysis: {r}" for t, r in zip(LLM_TEXTS_TR, RATIONALES["tr"])], dtype=object),
                          np.array([f"{t} | Analysis: {r}" for t, r in zip(LLM_TEXTS_DEV, RATIONALES["dev"])], dtype=object))
    for i in [0, 1, 2]:
        print(dev_df["tweet"][i][:150], "\n  →", RATIONALES["dev"][i], "\n")
    rat_kw = {**combo_kw, "prep": "rationale", "max_len": 256}
    run_config(make_cfg("rationale", **rat_kw))
    display(summary([n for n in [REFERENCE, "combo", "rationale"] if n in REG]))

@Fichinescu La comunidad gamer es un antro de misóginos y supremacistas blancos hace años pero hoy día pocos se acuerdan de gamergate y todo lo que ca 
  → The strongest argument that this tweet is sexist is that it describes a situation where the gaming community is characterized as misogynistic and racist, which can be seen as a criticism of sexist behaviors within that community. The strongest argument that it is not sexist is that the tweet is more focused on describing a historical context and the lasting impact of Gamergate, rather than directly insulting or objectifying women. 

@anacaotica88 @MordorLivin No me acuerdo de los detalles de GamerGate, pero ella estuvo en el ojo del huracán recibiendo acoso de hombres indignados ( 
  → Strongest argument that this tweet is sexist: The tweet reports a situation where a woman, likely a female gamer, was subjected to harassment and acrimony from men and their media supporters, which is a form of reported sexism as it describes a situat

,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
baseline,transformer,0.8450,0.27,0.8505,0.6054,0.8705,0.8678,0.8444,0.8896,0.8592,0.8743,0.5541,NaN,NaN
rationale,transformer,0.7607,0.45,0.7872,0.5648,0.8603,0.8728,0.8489,0.8939,0.8579,0.8839,0.5008,0.005,0.262


### 8.3 Fine-tuning LoRA du LLM, puis distillation vers XLM-R

Le LLM 7B devient un classifieur :

- **entrée** : une consigne courte (définition EXIST en une phrase) + le tweet, au format chat ;
- **sortie** : au premier token de la réponse, on ne garde que les logits de « NO » et « YES » → P(YES) par softmax sur ces deux logits ;
- **perte** : cross-entropy avec la cible soft [1 − p, p] ;
- **LoRA** : rang 16, α 32, dropout 0,05, sur toutes les projections d'attention et du MLP (≈40 M paramètres entraînables), poids du LLM gelés en bf16 (pas de quantification : 96 Go suffisent) ;
- **optimisation** : AdamW, LR 1e-4, warmup 5 %, décroissance linéaire, 2 époques, batch 16 ;
- **protocole** : les mêmes 5 folds ; l'époque est choisie sur l'OOF, et la prédiction du dev est la moyenne des 5 modèles.

Les adaptateurs sont retirés entre deux folds (`unload`) : le LLM n'est chargé qu'une fois.

**Distillation** : la prédiction OOF du LLM fine-tuné sert ensuite de seconde cible pour XLM-R (`distill` : moyenne de la part de YES et de la probabilité du LLM). XLM-R reste le modèle déployé, 15 fois plus petit. Attention : l'OOF de `distill` est un peu optimiste (la cible d'un tweet d'entraînement vient d'un LLM qui a vu le fold de validation) ; c'est le dev qui tranche pour cette ligne.

In [25]:
%pip install -U bitsandbytes>=0.46.1

In [26]:
FT_CFG = dict(epochs=2, lr=1e-4, bs=16, micro_bs=4, r=16, alpha=32, dropout=0.05, max_len=512, seed=SEED)   # lot effectif 16 = 4 micro-lots de 4
FT_TAG = f"llm_ft_{LLM_NAME.replace('/', '-')}{'_4bit' if LLM_FT_4BIT else ''}_{FT_CFG['epochs']}ep_lr{FT_CFG['lr']}_r{FT_CFG['r']}"
FT_INSTRUCTION = ("Following the EXIST 2024 guidelines (a tweet is sexist if it is sexist itself, reports a sexist situation "
                  "or criticises a sexist behaviour), is the following tweet sexist?\nTweet: {tweet}\nAnswer YES or NO.")


def ft_fold(fold, tok, base, enc_tr, enc_dev, yes_id, no_id):
    path = CACHE_DIR / "runs" / FT_TAG / f"fold{fold}.npz"
    if path.exists():
        return dict(np.load(path))
    path.parent.mkdir(parents=True, exist_ok=True)
    from peft import LoraConfig, get_peft_model
    set_seed(FT_CFG["seed"] * 100 + fold)
    rng = np.random.default_rng(FT_CFG["seed"] * 100 + fold)
    lora = LoraConfig(r=FT_CFG["r"], lora_alpha=FT_CFG["alpha"], lora_dropout=FT_CFG["dropout"], task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(base, lora)
    causal = model.get_base_model()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=FT_CFG["lr"], weight_decay=0.0)
    tr_idx, va_idx = np.where(FOLDS != fold)[0], np.where(FOLDS == fold)[0]
    steps = math.ceil(len(tr_idx) / FT_CFG["bs"]) * FT_CFG["epochs"]
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05 * steps), steps)

    @torch.no_grad()
    def predict_llm(encs):
        model.eval()
        order = np.argsort([-len(e) for e in encs]); out = np.zeros(len(encs))

        def step(idx):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                lg = last_token_logits(causal, *left_pad(tok, [encs[j] for j in idx]))[:, [no_id, yes_id]]
            out[idx] = torch.softmax(lg, 1)[:, 1].cpu().numpy()

        adaptive_batches(encs, order, LLM_TOKEN_BUDGET, step)
        return out

    p_va, p_dev, losses = [], [], []
    for epoch in range(FT_CFG["epochs"]):
        model.train()
        order = rng.permutation(tr_idx)
        ep = []
        for s in tqdm(range(0, len(order), FT_CFG["bs"]), desc=f"LLM LoRA fold {fold} époque {epoch + 1}", leave=False):
            idx = order[s:s + FT_CFG["bs"]]
            optimizer.zero_grad(set_to_none=True)
            batch_loss = 0.0
            for m0 in range(0, len(idx), FT_CFG["micro_bs"]):      # accumulation de gradient
                mi = idx[m0:m0 + FT_CFG["micro_bs"]]
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    lg = last_token_logits(causal, *left_pad(tok, [enc_tr[j] for j in mi]))[:, [no_id, yes_id]]
                p = torch.tensor(P_TR[mi], dtype=torch.float, device=device)
                loss = soft_cross_entropy(lg, torch.stack([1 - p, p], 1)) * len(mi) / len(idx)
                loss.backward()
                batch_loss += loss.item()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()
            ep.append(batch_loss)
        losses.append(np.mean(ep))
        p_va.append(predict_llm([enc_tr[j] for j in va_idx]))
        p_dev.append(predict_llm(enc_dev))
        print(f"  fold {fold} époque {epoch + 1} : perte {losses[-1]:.4f}, OOF F1 du fold {f1_at(Y_TR[va_idx][EVAL_TR[va_idx]], p_va[-1][EVAL_TR[va_idx]]):.4f}")
    res = dict(va_idx=va_idx, p_va=np.array(p_va), p_dev=np.array(p_dev), loss=np.array(losses))
    np.savez(path, **res)
    model.unload()                       # retire les adaptateurs : le LLM de base est réutilisé au fold suivant
    model = optimizer = scheduler = None
    gc.collect(); torch.cuda.empty_cache()
    return res


if RUN_LLM_FT:
    t0 = time.time()
    ft_tok, ft_base = None, None
    ft_dir = CACHE_DIR / "runs" / FT_TAG
    if not all((ft_dir / f"fold{k}.npz").exists() for k in range(N_FOLDS)):
        ft_tok, ft_base = load_llm(LLM_NAME, four_bit=LLM_FT_4BIT)
        ft_base.config.use_cache = False
        ft_base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    tok_for_prompts = ft_tok or AutoTokenizer.from_pretrained(LLM_NAME)
    yes_id = tok_for_prompts.encode("YES", add_special_tokens=False)[0]
    no_id = tok_for_prompts.encode("NO", add_special_tokens=False)[0]
    enc_ft = lambda texts: [tok_for_prompts(chat_text(tok_for_prompts, FT_INSTRUCTION.format(tweet=t[:700])), add_special_tokens=False,
                                            truncation=True, max_length=FT_CFG["max_len"])["input_ids"] for t in texts]
    ENC_FT_TR, ENC_FT_DEV = enc_ft(LLM_TEXTS_TR), enc_ft(LLM_TEXTS_DEV)
    if ft_tok is not None:
        ft_tok.padding_side = "left"
    oof, dev = collect_folds(lambda k: ft_fold(k, tok_for_prompts, ft_base, ENC_FT_TR, ENC_FT_DEV, yes_id, no_id), FT_CFG["epochs"])
    select_epoch("llm_ft_lora", oof, dev, "LLM fine-tuné", model=LLM_NAME)
    if ft_base is not None:
        del ft_base; gc.collect(); torch.cuda.empty_cache()
    print(f"({(time.time() - t0) / 60:.1f} min)")

    # Distillation vers XLM-R
    SOFT_TARGETS["distill"] = 0.5 * (P_TR + REG["llm_ft_lora"]["oof"])
    run_config(make_cfg("distill", **with_target(combo_kw, "distill")))
    REG["distill"]["optimistic"] = True      # exclu des choix automatiques (ensemble, modèle final) : jugé sur le dev
    display(summary([n for n in [REFERENCE, "combo", "combo_3seeds", "llm_ft_lora", "distill"] if n in REG]))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

... (55 outputs left out)

distill fold 4 époque 2:   0%|          | 0/346 [00:00<?, ?it/s]

distill fold 4 époque 3:   0%|          | 0/346 [00:00<?, ?it/s]

distill                époque retenue 3 | OOF F1 0.8622 | dev F1@0.5 0.8772
   (3.9 min)


,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
distill,transformer,0.8573,0.45,0.8622,0.5137,0.8772,0.8782,0.8550,0.8989,0.8593,0.8927,0.4934,0.0104,0.0545
llm_ft_lora,LLM fine-tuné,0.8613,0.47,0.8613,0.5112,0.8705,0.8772,0.8524,0.8968,0.8571,0.8923,0.4887,0.0093,0.1905
baseline,transformer,0.8450,0.27,0.8505,0.6054,0.8705,0.8678,0.8444,0.8896,0.8592,0.8743,0.5541,NaN,NaN


## 9. Combinaisons

Toutes les combinaisons sont apprises **sur l'OOF** et appliquées telles quelles au dev.

1. **Ensemble glouton (Caruana)** : on ajoute, avec remise, le run qui améliore le plus l'OOF F1 de la moyenne ; une fois parmi les transformers seuls (`ens_glouton`, notre « BERT » pour la suite), une fois en autorisant le LLM fine-tuné (`ens_glouton+llm_ft`).
2. **Stacking** (régression logistique, et gradient boosting) : BERT seul (calibration + langue), BERT + modèle 1, BERT + LLM (few-shot et fine-tuné), tout.
3. **Cascade** (l'idée « le premier modèle décide de façon déterministe quand il est sûr, sinon BERT ») : si le modèle 1 donne p < τ ou p > 1 − τ, on garde sa décision ; sinon celle de BERT. τ choisi sur l'OOF ; on affiche la part des tweets décidés par le modèle 1.
4. **Règle lexique** : « mot du lexique sexiste ⇒ YES », appliquée par-dessus BERT.
5. **Routage vers le LLM** (« When in doubt, consult ») : si BERT est incertain (|p − 0,5| < δ après calibration), on prend la moyenne des probabilités calibrées de BERT et du meilleur LLM (selon l'OOF) ; δ choisi sur l'OOF.
6. **Seuil par langue** (l'anglais est nettement plus bas que l'espagnol).

In [27]:
def greedy_ensemble(names, n_iter=15):
    chosen, best = [], -1
    for _ in range(n_iter):
        scores = {n: best_threshold(np.mean([REG[c]["oof"] for c in chosen + [n]], 0))[1] for n in names}
        n_best = max(scores, key=scores.get)
        if scores[n_best] <= best + 1e-5:
            break
        chosen.append(n_best); best = scores[n_best]
    return chosen


TRANSFORMER_RUNS = [n for n, r in REG.items() if r["kind"] == "transformer" and not r.get("optimistic")
                    and not n.startswith("sched_")]   # les calendriers non retenus sont exclus ; base_* / baseline restent
LLM_FT_RUNS = [n for n, r in REG.items() if r["kind"] == "LLM fine-tuné"]
if TRANSFORMER_RUNS:
    ENS = greedy_ensemble(TRANSFORMER_RUNS)
    print("Ensemble glouton (transformers) :", Counter(ENS))
    register("ens_glouton", np.mean([REG[n]["oof"] for n in ENS], 0), np.mean([REG[n]["dev"] for n in ENS], 0),
             "combinaison", inputs=ENS)
    if LLM_FT_RUNS:
        ENS_ALL = greedy_ensemble(TRANSFORMER_RUNS + LLM_FT_RUNS)
        print("Ensemble glouton (transformers + LLM fine-tuné) :", Counter(ENS_ALL))
        register("ens_glouton+llm_ft", np.mean([REG[n]["oof"] for n in ENS_ALL], 0),
                 np.mean([REG[n]["dev"] for n in ENS_ALL], 0), "combinaison", inputs=ENS_ALL)
    BERT = "ens_glouton"

Ensemble glouton (transformers) : Counter({'combo_seeds': 1, 'rationale': 1, 'multitask': 1})
Ensemble glouton (transformers + LLM fine-tuné) : Counter({'llm_ft_lora': 1, 'soft_s43': 1, 'soft_s42': 1})


In [28]:
LLM_AVAILABLE = [n for n, r in REG.items() if r["kind"] in ("LLM", "LLM fine-tuné")]
STACKS = {
    "stack_bert": [BERT],
    "stack_bert+modele1": [BERT] + MODEL1_INPUTS,
    "stack_bert+llm": [BERT] + LLM_AVAILABLE,
    "stack_tout": [BERT] + MODEL1_INPUTS + LLM_AVAILABLE,
}
for name, inputs in STACKS.items():
    if all(n in REG for n in inputs) and (name != "stack_bert+llm" or LLM_AVAILABLE):
        fit_stack(name, inputs)
if all(n in REG for n in STACKS["stack_tout"]):
    fit_stack("stack_tout+features", STACKS["stack_tout"], with_hand=True)
    fit_stack("stack_tout_hgb", STACKS["stack_tout"], with_hand=True, model="hgb")
summary([n for n in REG if REG[n]["kind"] == "combinaison"] + [BERT])

,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
stack_tout,combinaison,0.8684,0.47,0.8708,0.6072,0.8822,0.8825,0.8588,0.9026,0.8579,0.9010,0.5730,0.0146,0.0505
stack_bert+llm,combinaison,0.8681,0.47,0.8686,0.6063,0.8792,0.8837,0.8605,0.9035,0.8601,0.9014,0.5678,0.0159,0.0335
stack_tout+features,combinaison,0.8656,0.46,0.8685,0.6113,0.8866,0.8891,0.8655,0.9089,0.8682,0.9049,0.5747,0.0213,0.0100
ens_glouton+llm_ft,combinaison,0.8649,0.48,0.8671,0.5096,0.8850,0.8882,0.8650,0.9085,0.8680,0.9035,0.4871,0.0203,0.0070
stack_tout_hgb,combinaison,0.8591,0.36,0.8616,0.6652,0.8742,0.8810,0.8571,0.9007,0.8564,0.8994,0.6143,0.0131,0.0625
stack_bert+modele1,combinaison,0.8585,0.51,0.8601,0.6089,0.8749,0.8723,0.8480,0.8941,0.8382,0.8976,0.5696,0.0045,0.3305
stack_bert,combinaison,0.8536,0.42,0.8598,0.6129,0.8767,0.8780,0.8549,0.8995,0.8520,0.8973,0.5579,0.0101,0.0935
ens_glouton,combinaison,0.8558,0.47,0.8595,0.5235,0.8780,0.8765,0.8539,0.8977,0.8550,0.8927,0.4943,0.0087,0.1250
modele1,combinaison,0.8250,0.35,0.8329,0.6107,0.8288,0.8413,0.8154,0.8659,0.8235,0.8540,0.5974,-0.0265,0.9945


### 9.1 Cascade : le modèle 1 décide quand il est sûr

In [29]:
def cascade(p1, p2, thr1, thr2, tau):
    sure = (p1 < tau) | (p1 > 1 - tau)
    return np.where(sure, (p1 >= thr1).astype(float), (p2 >= thr2).astype(float)), sure


p1, p2 = REG["modele1"], REG[BERT]
thr1, _ = best_threshold(p1["oof"]); thr2, _ = best_threshold(p2["oof"])
rows = []
for tau in [0.0, 0.02, 0.05, 0.08, 0.10, 0.15, 0.20]:
    d_oof, s_oof = cascade(p1["oof"], p2["oof"], thr1, thr2, tau)
    d_dev, s_dev = cascade(p1["dev"], p2["dev"], thr1, thr2, tau)
    rows.append({"τ": tau, "part décidée par le modèle 1 (OOF)": s_oof[EVAL_TR].mean(),
                 "exactitude modèle 1 sur ces tweets": ((p1["oof"] >= thr1) == Y_TR)[EVAL_TR & s_oof].mean() if (EVAL_TR & s_oof).any() else np.nan,
                 "exactitude BERT sur ces tweets": ((p2["oof"] >= thr2) == Y_TR)[EVAL_TR & s_oof].mean() if (EVAL_TR & s_oof).any() else np.nan,
                 "OOF F1": f1_at(Y_TR[EVAL_TR], d_oof[EVAL_TR]), "dev F1": f1_at(Y_DEV[EVAL_DEV], d_dev[EVAL_DEV])})
cascade_table = pd.DataFrame(rows).round(4)
display(cascade_table)
tau_best = cascade_table.loc[cascade_table["OOF F1"].idxmax(), "τ"]
d_oof, _ = cascade(p1["oof"], p2["oof"], thr1, thr2, tau_best)
d_dev, _ = cascade(p1["dev"], p2["dev"], thr1, thr2, tau_best)
register("cascade_modele1_bert", d_oof * 0.98 + 0.01, d_dev * 0.98 + 0.01, "combinaison", tau=tau_best)
print("τ retenu sur l'OOF :", tau_best)

,τ,part décidée par le modèle 1 (OOF),exactitude modèle 1 sur ces tweets,exactitude BERT sur ces tweets,OOF F1,dev F1
0,0.00,0.0000,NaN,NaN,0.8595,0.8765
1,0.02,0.1718,0.9866,0.9866,0.8595,0.8765
2,0.05,0.3331,0.9792,0.9777,0.8600,0.8775
3,0.08,0.4321,0.9653,0.9637,0.8600,0.8782
4,0.10,0.4847,0.9592,0.9585,0.8594,0.8791
5,0.15,0.5909,0.9436,0.9431,0.8591,0.8798
6,0.20,0.6761,0.9256,0.9261,0.8577,0.8764


τ retenu sur l'OOF : 0.05


### 9.2 Règle lexique et routage des cas incertains vers le LLM

In [30]:
LEX_HIT_TR = HF_TR[:, FEATURE_NAMES.index("lex_sexist")] > 0
LEX_HIT_DEV = HF_DEV[:, FEATURE_NAMES.index("lex_sexist")] > 0
thr_b, _ = best_threshold(REG[BERT]["oof"])
print(f"Règle lexique : se déclenche sur {LEX_HIT_TR.mean():.1%} du train, précision {Y_TR[EVAL_TR & LEX_HIT_TR].mean():.3f} ; "
      f"FN de BERT corrigés (OOF) : {(EVAL_TR & LEX_HIT_TR & (REG[BERT]['oof'] < thr_b) & (Y_TR == 1)).sum()}, "
      f"FP ajoutés : {(EVAL_TR & LEX_HIT_TR & (REG[BERT]['oof'] < thr_b) & (Y_TR == 0)).sum()}")
register("bert+regle_lexique",
         np.where(LEX_HIT_TR, np.maximum(REG[BERT]["oof"], thr_b + 0.01), REG[BERT]["oof"]),
         np.where(LEX_HIT_DEV, np.maximum(REG[BERT]["dev"], thr_b + 0.01), REG[BERT]["dev"]), "combinaison")


def calibrate(name):
    """Calibration de Platt apprise sur l'OOF (non-égalités), en validation croisée pour l'OOF."""
    x_tr, x_dev = logit(REG[name]["oof"]).reshape(-1, 1), logit(REG[name]["dev"]).reshape(-1, 1)
    oof = np.zeros(N_TR)
    for k in range(N_FOLDS):
        tr, va = EVAL_TR & (FOLDS != k), FOLDS == k
        oof[va] = LogisticRegression().fit(x_tr[tr], Y_TR[tr]).predict_proba(x_tr[va])[:, 1]
    dev = LogisticRegression().fit(x_tr[EVAL_TR], Y_TR[EVAL_TR]).predict_proba(x_dev)[:, 1]
    return oof, dev


if LLM_AVAILABLE:
    ROUTE_LLM = max(LLM_AVAILABLE, key=oof_score)
    print("LLM consulté pour les cas incertains :", ROUTE_LLM)
    cb_oof, cb_dev = calibrate(BERT)
    cl_oof, cl_dev = calibrate(ROUTE_LLM)
    rows = []
    for delta in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.50]:
        r_oof = np.where(np.abs(cb_oof - 0.5) < delta, (cb_oof + cl_oof) / 2, cb_oof)
        r_dev = np.where(np.abs(cb_dev - 0.5) < delta, (cb_dev + cl_dev) / 2, cb_dev)
        rows.append({"δ": delta, "part routée (dev)": (np.abs(cb_dev - 0.5) < delta).mean(),
                     "OOF F1@seuil": best_threshold(r_oof)[1], "dev F1@seuil OOF": f1_at(Y_DEV[EVAL_DEV], r_dev[EVAL_DEV], best_threshold(r_oof)[0])})
    routing = pd.DataFrame(rows).round(4)
    display(routing)
    d = routing.loc[routing["OOF F1@seuil"].idxmax(), "δ"]
    register("routage_llm", np.where(np.abs(cb_oof - 0.5) < d, (cb_oof + cl_oof) / 2, cb_oof),
             np.where(np.abs(cb_dev - 0.5) < d, (cb_dev + cl_dev) / 2, cb_dev), "combinaison", delta=d)
    print("δ retenu sur l'OOF :", d)

Règle lexique : se déclenche sur 6.9% du train, précision 0.870 ; FN de BERT corrigés (OOF) : 15, FP ajoutés : 26
LLM consulté pour les cas incertains : llm_ft_lora


,δ,part routée (dev),OOF F1@seuil,dev F1@seuil OOF
0,0.00,0.0000,0.8598,0.8777
1,0.05,0.0414,0.8623,0.8818
2,0.10,0.0886,0.8625,0.8800
3,0.15,0.1339,0.8647,0.8793
4,0.20,0.1811,0.8680,0.8840
5,0.25,0.2283,0.8683,0.8840
6,0.30,0.2823,0.8682,0.8840
7,0.50,1.0000,0.8675,0.8864


δ retenu sur l'OOF : 0.25


### 9.3 Seuil par langue

In [31]:
def per_language_threshold(name):
    r = REG[name]
    pred_dev = np.zeros(N_DEV)
    thr = {}
    for lang in ["en", "es"]:
        m_tr = EVAL_TR & (LANG_TR == lang)
        scores = [f1_at(Y_TR[m_tr], r["oof"][m_tr], t) for t in THR_GRID]
        thr[lang] = float(THR_GRID[int(np.argmax(scores))])
        pred_dev[LANG_DEV == lang] = r["dev"][LANG_DEV == lang] >= thr[lang]
    return thr, f1_at(Y_DEV[EVAL_DEV], pred_dev[EVAL_DEV])


rows = []
for name in [BERT, "ens_glouton+llm_ft", "stack_tout", "stack_bert+modele1", "routage_llm", "llm_ft_lora"]:
    if name in REG:
        thr, f1_dev = per_language_threshold(name)
        rows.append({"modèle": name, "seuil EN": thr["en"], "seuil ES": thr["es"],
                     "dev F1 seuil par langue": f1_dev, "dev F1 seuil unique": evaluate(name)["dev F1@seuil"]})
pd.DataFrame(rows).set_index("modèle").round(4)

,seuil EN,seuil ES,dev F1 seuil par langue,dev F1 seuil unique
modèle,,,,
ens_glouton,0.47,0.47,0.8765,0.8765
ens_glouton+llm_ft,0.48,0.48,0.8882,0.8882
stack_tout,0.47,0.44,0.8842,0.8825
stack_bert+modele1,0.40,0.45,0.8813,0.8723
routage_llm,0.41,0.44,0.8915,0.8840
llm_ft_lora,0.47,0.47,0.8772,0.8772


## 10. Tableau final

Tous les composants, triés par **OOF F1 au seuil optimal** (le critère de choix). Les colonnes dev sont la mesure finale : intervalle bootstrap à 95 %, écart avec la **baseline** (XLM-R large, moyenne hard + soft) et proportion de rééchantillonnages où cet écart est ≤ 0 (une valeur sous 0,05 indique un gain significatif).

In [32]:
final_table = summary()
final_table.to_csv(RESULTS_CSV)
print("Tableau enregistré dans", RESULTS_CSV)
final_table

Tableau enregistré dans /content/drive/MyDrive/TweetClassification/resultats_modelling2.csv


,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
stack_tout,combinaison,0.8684,0.47,0.8708,0.6072,0.8822,0.8825,0.8588,0.9026,0.8579,0.9010,0.5730,0.0146,0.0505
stack_bert+llm,combinaison,0.8681,0.47,0.8686,0.6063,0.8792,0.8837,0.8605,0.9035,0.8601,0.9014,0.5678,0.0159,0.0335
stack_tout+features,combinaison,0.8656,0.46,0.8685,0.6113,0.8866,0.8891,0.8655,0.9089,0.8682,0.9049,0.5747,0.0213,0.0100
routage_llm,combinaison,0.8665,0.47,0.8683,0.6138,0.8807,0.8840,0.8612,0.9039,0.8667,0.8971,0.5576,0.0161,0.0330
ens_glouton+llm_ft,combinaison,0.8649,0.48,0.8671,0.5096,0.8850,0.8882,0.8650,0.9085,0.8680,0.9035,0.4871,0.0203,0.0070
distill,transformer,0.8573,0.45,0.8622,0.5137,0.8772,0.8782,0.8550,0.8989,0.8593,0.8927,0.4934,0.0104,0.0545
stack_tout_hgb,combinaison,0.8591,0.36,0.8616,0.6652,0.8742,0.8810,0.8571,0.9007,0.8564,0.8994,0.6143,0.0131,0.0625
llm_ft_lora,LLM fine-tuné,0.8613,0.47,0.8613,0.5112,0.8705,0.8772,0.8524,0.8968,0.8571,0.8923,0.4887,0.0093,0.1905
stack_bert+modele1,combinaison,0.8585,0.51,0.8601,0.6089,0.8749,0.8723,0.8480,0.8941,0.8382,0.8976,0.5696,0.0045,0.3305


**Modèle retenu** : le meilleur composant selon l'OOF (jamais selon le dev). On affiche son rapport de classification, ses erreurs par nombre de votes, et on exporte ses prédictions pour l'itération suivante.

In [33]:
CANDIDATES = [n for n in REG if REG[n]["kind"] in ("transformer", "combinaison", "LLM", "LLM fine-tuné") and not REG[n].get("optimistic")]
FINAL = max(CANDIDATES, key=oof_score)
thr_final, _ = best_threshold(REG[FINAL]["oof"])
p_final = REG[FINAL]["dev"]
pred_final = (p_final >= thr_final).astype(int)
print(f"Modèle retenu : {FINAL} | seuil {thr_final} | dev F1 = {f1_at(Y_DEV[EVAL_DEV], p_final[EVAL_DEV], thr_final):.4f}")
if REFERENCE in REG:
    r = evaluate(FINAL)
    print(f"Baseline : dev F1 = {evaluate(REFERENCE)['dev F1@seuil']:.4f} | gain = {r.get('Δ dev vs réf', 0):+.4f} "
          f"(p(Δ≤0) = {r.get('p(Δ≤0)', float('nan')):.3f})")

from sklearn.metrics import classification_report
print(classification_report(Y_DEV[EVAL_DEV], pred_final[EVAL_DEV], target_names=["NO", "YES"], digits=3))

err = pd.DataFrame({"n_yes": dev_df["n_yes"], "ok": pred_final == Y_DEV})[EVAL_DEV]
display(err.groupby("n_yes")["ok"].agg(["count", "mean"]).rename(columns={"mean": "exactitude"}).round(3))

Modèle retenu : stack_tout | seuil 0.47 | dev F1 = 0.8825
Baseline : dev F1 = 0.8678 | gain = +0.0146 (p(Δ≤0) = 0.051)
              precision    recall  f1-score   support

          NO      0.883     0.898     0.890       479
         YES      0.890     0.875     0.882       455

    accuracy                          0.887       934
   macro avg      0.887     0.886     0.886       934
weighted avg      0.887     0.887     0.886       934



,count,exactitude
n_yes,,
0,203,1.000
1,160,0.912
2,116,0.698
4,144,0.729
5,148,0.905
6,163,0.975


In [34]:
out = dev_df[["id_EXIST", "lang", "tweet", "n_yes", "p_yes", "hard"]].copy()
out["tweet_v2"] = TEXTS["v2"][1]
out["mot_cle"] = SEED_DEV
for name in REG:
    out[f"p_{name}"] = REG[name]["dev"]
out["pred"] = np.where(pred_final == 1, "YES", "NO")
out["error_type"] = np.select([EVAL_DEV & (pred_final == 1) & (Y_DEV == 0), EVAL_DEV & (pred_final == 0) & (Y_DEV == 1)], ["FP", "FN"], default="")
out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"{len(out)} tweets enregistrés dans {OUTPUT_CSV} ; erreurs : {out['error_type'].value_counts().to_dict()}")

print("\nFaux négatifs les plus nets :")
display(out[out["error_type"] == "FN"].sort_values(f"p_{FINAL}")[["lang", "tweet", "n_yes", f"p_{FINAL}", "mot_cle"]].head(15))
print("Faux positifs les plus nets :")
display(out[out["error_type"] == "FP"].sort_values(f"p_{FINAL}", ascending=False)[["lang", "tweet", "n_yes", f"p_{FINAL}", "mot_cle"]].head(15))

1038 tweets enregistrés dans /content/drive/MyDrive/TweetClassification/dev_predictions_v2.csv ; erreurs : {'': 932, 'FN': 57, 'FP': 49}

Faux négatifs les plus nets :


,lang,tweet,n_yes,p_stack_tout,mot_cle
35,es,"Absuelto, si, pero, quién le devuelve su vida?. Cuanto daño el pvto #YoSiTeCreo @IreneMontero @Adrilastra #ComunismoEsMiseria ...",4,0.014079,creo yositecreo
82,es,"#EnVivo Conferencia ""Barreras invisibles para madres trabajadoras"", impartida por la Mtra. Virginia Shaday Gutiérrez León https://t.co/a...",5,0.043031,barreras
219,es,"👉🏼Corrección: La @UESoficial, pese que es una institución pública, no cuenta con Unidad de Género. La instancia a la que señalamos de in...",4,0.057381,estudios de genero
912,en,"@djxmenez yea back when i didn’t even have one to slap, unfortunate !",4,0.068903,slap
660,en,@Lucesheffield @JuliaPa28346879 What is 'difference feminism'?? Never heard of this before.,4,0.069722,difference feminism
149,es,"@Bautiicapitani Uno te pregunta preocupado, y el culo parado no saluda por la calle. Mala persona",4,0.075478,culo
446,es,"El lobo Carrasco si puede le come la polla a Messi a dos manos, que cansino",4,0.090679,polla
112,es,con q motivación voy a entrenar ahora si ya no veo a ninguna de las chicas 😭,5,0.098825,chicas
860,en,"Cyborg politics is the struggle for language and the struggle against perfect communication, against the one code that translates all me...",5,0.100120,phallogocentrism
592,en,@5heriBr0wn @stinkythinktank @nubbin00_ @MeganMcD411 @wellroundednut @frggystyl @HauntedJennifer @normhotchkiss At first I was scared of...,4,0.106965,as good as a


Faux positifs les plus nets :


,lang,tweet,n_yes,p_stack_tout,mot_cle
785,en,@BoopBepp @Helloth07860784 @Bronwen85 Yup I hate when men rape and kill women. https://t.co/MyfVPu6THd,2,0.958797,kill women
973,en,@bhargavtata0 oh so now u can’t understand women’s words??????,2,0.930094,understand women
190,es,@Cristina_Coach @Dillita1 @lapropriaWinter @Alex_con_3X ¿Usted acaso vio a Ruiz Cortines saliendo a militar con movimientos feministas? ...,2,0.919163,las mujeres no
1029,en,"Uh, they're allowed to? calling this out makes you look like a cunt. Fortunately, I'll block you so fast you wont get your screenshot. h...",2,0.912714,you look like a
358,es,@clitfreaks @Andres54167596 Soy muy cortado aunque me gusta masturbarme viendo como se follan por el culo todavia soy virgen me gustaria...,1,0.894306,me gustaria follar
549,en,"@Mike_Fabricant “You should smile more, love. Just pretend you’re a Tory opening a food bank. We’re only living through the worst #CostO...",2,0.893857,everyday sexism
314,es,"@IreneMontero @IreneMontero, efectivamente.. las mujeres no deben de ser discriminadas salarialmente por pertenecer a cuerpos policiales...",2,0.892549,las mujeres no
451,es,"@PSOE @Adrilastra La prostitución forzada, el proxenetismo, el rufianismo, la trata o como queráis llamarlo, ya está contemplado en el C...",2,0.885333,forzada
476,es,"Siempre y lo será una mierda, con su transfobia, machismo, homofobia, sexismo, xenofobia, y los personajes insoportables, y la gente tie...",2,0.878178,sexismo
791,en,"Ladies, don't have a miscarriage in Louisiana! If you do, in addition to probably having your friends and family sued by some opportunis...",2,0.857338,ladies don t


## 11. Lecture des résultats

**Comment lire le tableau.** Une idée n'est retenue que si elle bat la baseline sur l'OOF (6 000 tweets, critère de choix) **et** sur le dev, avec p(Δ≤0) < 0,05. Les questions posées, et les lignes à comparer :

| Question | Lignes |
|---|---|
| Quel calendrier pour la baseline ? | `sched_*` (partie 6.1) |
| Le preprocessing corrigé aide-t-il ? | `prep_v2` vs `base_soft` |
| AEDA ? | `aeda` vs `base_soft` |
| **Multi-classes vs binaire** ? | `cls4`, `cls7`, `multitask` vs `base_soft` (+ tableau par intention, partie 6) |
| **Tête CNN vs linéaire** ? | `head_cnn` vs `base_soft` |
| Meilleurs réglages + 3 seeds ? | `combo_pipeline` vs `baseline` |
| **XLM-R XL** ? | `xl_pipeline` vs `combo_pipeline` et `baseline` |
| **Premier modèle + BERT** ? | `stack_bert+modele1`, `cascade_modele1_bert`, `bert+regle_lexique` vs `ens_glouton` |
| **LLM few-shot** ? | `llm_few_shot` seul ; `stack_bert+llm`, `routage_llm` vs `ens_glouton` |
| **Justifications LLM** ? | `rationale` vs `combo` |
| **LLM fine-tuné (LoRA)** ? | `llm_ft_lora` seul ; `ens_glouton+llm_ft` vs `ens_glouton` |
| Distillation ? | `distill` vs `combo` (sur le dev uniquement) |
| Seuil par langue ? | tableau 10.3 |

La ligne « Modèle retenu » (partie 11) donne la meilleure pipeline obtenue et son gain sur la baseline.